<div align="center">

#
## MolFlood - Docking Score Prediction Pipeline

**Version:** 0.9.0 — Public Beta  
**Predictive endpoint:** docking score  
**Focus:** reproducible ML-assisted molecular prioritization, with emphasis on understudied and neglected disease targets.

**Developed by:**  
**Davidt Tarouco** · **Dr. Conrado Pedebos** · **Dr. Rodrigo Ligabue-Braun**

</div>

---

MolFlood is a reproducible machine-learning workflow for learning and predicting docking scores from molecular structure. It combines Morgan fingerprints, RDKit physicochemical descriptors, structural validation, hyperparameter optimization, applicability-domain analysis, interpretability and external molecular prediction.

> **Scientific scope:** MolFlood predicts docking scores produced within a defined computational docking context. It does not directly predict experimental binding affinity, biological activity or therapeutic efficacy.

### Public-beta status

This notebook is released as **MolFlood v0.9.0 Public Beta**. A tutorial dataset prepared by the developers may be distributed with the public release so users can reproduce the complete workflow before applying MolFlood to their own docking campaigns.


## Scientific workflow overview

MolFlood uses publication-oriented validation:

1. `docking_score` is the single predictive endpoint.
2. The held-out test set is not used for algorithm selection or Optuna.
3. Model selection and hyperparameter optimization use training data only.
4. Structural split modes use group-aware cross-validation.
5. External predictions include Morgan/Tanimoto applicability-domain information.
6. Seeds, dependency versions, dataset hashes, split settings and model settings are recorded.
7. External predictions include docking-score predictions and applicability-domain information.

The training dataset is assumed to have been deduplicated before entering this notebook.

In [ ]:
# MolFlood v0.9.0 dependency bootstrap.
# Recommended Python: 3.12-3.14.
#
# Run this cell before the scientific workflow. It installs exact versions only
# when the current environment differs from the MolFlood requirements snapshot.

from pathlib import Path
from importlib import metadata as _metadata
import subprocess
import sys

MOLFLOOD_REQUIREMENTS_TEXT = """numpy==2.5.2
pandas==3.0.5
scipy==1.18.0
matplotlib==3.11.1
seaborn==0.13.2
scikit-learn==1.9.0
rdkit==2026.3.5
xgboost==3.4.1
lightgbm==4.7.0
catboost==1.2.10
optuna==4.9.0
joblib==1.5.3
shap==0.52.0
tabulate==0.10.0
"""

REQUIREMENTS_FILE = Path.cwd() / "requirements.txt"

if not REQUIREMENTS_FILE.exists():
    REQUIREMENTS_FILE.write_text(
        "# MolFlood v0.9.0 Public Beta\n"
        "# Current stable runtime pins selected on 2026-08-24.\n"
        "# Recommended Python: 3.12-3.14.\n\n"
        + MOLFLOOD_REQUIREMENTS_TEXT,
        encoding="utf-8",
    )
    print("Created:", REQUIREMENTS_FILE.resolve())

_required = {}
for _line in MOLFLOOD_REQUIREMENTS_TEXT.splitlines():
    if "==" in _line:
        _pkg, _ver = _line.split("==", 1)
        _required[_pkg.strip()] = _ver.strip()

def _installed_version(name):
    try:
        return _metadata.version(name)
    except _metadata.PackageNotFoundError:
        return None

_mismatches = {
    pkg: (_installed_version(pkg), required)
    for pkg, required in _required.items()
    if _installed_version(pkg) != required
}

if _mismatches:
    print("Installing/updating pinned MolFlood dependencies...")
    for _pkg, (_current, _expected) in _mismatches.items():
        print(f"  {_pkg}: installed={_current!r}; required={_expected}")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-r", str(REQUIREMENTS_FILE)]
    )
    print(
        "\nDependency installation completed. If your notebook platform requests "
        "a kernel restart, restart once and run the notebook again."
    )
else:
    print("All pinned MolFlood dependencies are already installed.")

## USER CONFIGURATION — edit this section first

For normal external use, **this is the main cell that should be edited**.

Disease, biological-target, protein-structure and docking-protocol fields are provenance metadata used in the Model Card and saved artifacts. They are not supplied to the molecular regression model as features.

MolFlood v0.9.0 intentionally exposes one publication-grade validation workflow rather than reduced-validation modes.

In [ ]:
#@title USER CONFIGURATION — edit this cell

# ------------------------------------------------------------------
# MolFlood release identity — do not edit for this release
# ------------------------------------------------------------------
MOLFLOOD_VERSION = "0.9.0"
MOLFLOOD_RELEASE_STATUS = "Public Beta"
MOLFLOOD_AUTHORS = [
    "Davidt Tarouco",
    "Dr. Conrado Pedebos",
    "Dr. Rodrigo Ligabue-Braun",
]
PUBLICATION_MODE = True

# ------------------------------------------------------------------
# Project folder
# ------------------------------------------------------------------
PROJECT_FOLDER = "dockml_run" #@param {type:"string"}

# ------------------------------------------------------------------
# Campaign / disease / biological target
# Metadata only: these fields are NOT ML features.
# ------------------------------------------------------------------
CAMPAIGN_ID = "replace_with_campaign_id" #@param {type:"string"}
CAMPAIGN_TITLE = "Docking score prediction campaign" #@param {type:"string"}

DISEASE_NAME = "replace_with_disease_name" #@param {type:"string"}
DISEASE_CATEGORY = "understudied_or_neglected_disease" #@param {type:"string"}
DISEASE_CONTEXT = "Briefly describe why this disease/therapeutic area is being investigated." #@param {type:"string"}

TARGET_NAME = "replace_with_target_name" #@param {type:"string"}
TARGET_GENE = None
TARGET_UNIPROT_ID = None
TARGET_ORGANISM = "replace_with_organism" #@param {type:"string"}
TARGET_DESCRIPTION = "Brief description of the biological target." #@param {type:"string"}

# ------------------------------------------------------------------
# Protein structure
# ------------------------------------------------------------------
STRUCTURE_SOURCE = "PDB"
PDB_ID = None
PROTEIN_CHAIN = None
STRUCTURE_NOTES = None

# ------------------------------------------------------------------
# Docking provenance
# ------------------------------------------------------------------
DOCKING_SOFTWARE = "replace_with_docking_software" #@param {type:"string"}
DOCKING_SOFTWARE_VERSION = None
DOCKING_PROTOCOL_ID = "replace_with_protocol_id" #@param {type:"string"}

BINDING_SITE_DEFINITION = None
GRID_CENTER = None
GRID_SIZE = None
EXHAUSTIVENESS = None
LIGAND_PREPARATION = None
PROTEIN_PREPARATION = None

# ------------------------------------------------------------------
# Structural validation
# ------------------------------------------------------------------
# fingerprint_cluster is mapped internally to morgan_kmeans_cluster.
SPLIT_MODE_USER = "fingerprint_cluster" #@param ["random_stratified", "scaffold", "fingerprint_cluster"]

# ------------------------------------------------------------------
# Optional external prediction
# ------------------------------------------------------------------
RUN_EXTERNAL_PREDICTION = False #@param {type:"boolean"}

# ------------------------------------------------------------------
# Publication workload
# ------------------------------------------------------------------
OPTUNA_TRIALS = 75
BOOTSTRAP_REPEATS = 2000
Y_SCRAMBLING_REPEATS = 30

# Estimators may parallelize internally. Cross-validation remains single-level
# to avoid nested CPU oversubscription.
REPRODUCIBLE_MODE = True
MODEL_N_JOBS = -1
CV_N_JOBS = 1

print("=" * 72)
print(f"MolFlood v{MOLFLOOD_VERSION} — {MOLFLOOD_RELEASE_STATUS}")
print("=" * 72)
print("Project folder:", PROJECT_FOLDER)
print("Disease:", DISEASE_NAME)
print("Target:", TARGET_NAME)
print("Structural split:", SPLIT_MODE_USER)
print("External prediction:", RUN_EXTERNAL_PREDICTION)
print("Optuna trials:", OPTUNA_TRIALS)
print("Bootstrap repeats:", BOOTSTRAP_REPEATS)
print("Y-scrambling repeats:", Y_SCRAMBLING_REPEATS)
print("=" * 72)

## 2. Imports and configuration

In [ ]:
import os
import sys
import json
import random
import hashlib
import platform
import warnings
from importlib import metadata

# Keep scientific/runtime warnings visible in publication mode.
warnings.filterwarnings("default")

import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, AllChem, DataStructs, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.dummy import DummyRegressor
from sklearn.base import clone
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, GroupKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.cluster import MiniBatchKMeans
from scipy.stats import spearmanr

from xgboost import XGBRegressor

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

try:
    from catboost import CatBoostRegressor
except ImportError:
    CatBoostRegressor = None

RANDOM_STATE = 123
CV_FOLDS = 5
BOOTSTRAP_CONFIDENCE = 0.95

N_JOBS = CV_N_JOBS

os.environ["PYTHONHASHSEED"] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")

print("Random seed:", RANDOM_STATE)
print("Publication mode:", PUBLICATION_MODE)
print("Reproducible mode:", REPRODUCIBLE_MODE)
print("Cross-validation jobs:", CV_N_JOBS)
print("Estimator jobs:", MODEL_N_JOBS)

In [ ]:
def require_publication_dependencies():
    missing = []

    if LGBMRegressor is None:
        missing.append("lightgbm")
    if CatBoostRegressor is None:
        missing.append("catboost")

    for package_name in ("shap",):
        try:
            __import__(package_name)
        except Exception:
            missing.append(package_name)

    if PUBLICATION_MODE and missing:
        raise ImportError(
            "PUBLICATION_MODE=True requires the complete expected dependency set. "
            f"Missing/incompatible packages: {sorted(set(missing))}."
        )

    print("Publication dependency check: OK" if not missing else f"Optional packages missing: {missing}")

require_publication_dependencies()

## 2.1 Project paths

By default, MolFlood stores inputs and outputs inside the folder defined by `PROJECT_FOLDER` in **USER CONFIGURATION**.

For HPC/server workflows, the environment variable `DOCKML_BASE_DIR` may override this location without modifying the notebook.

In [ ]:
from pathlib import Path

# Portable default: create/use a project folder beside the notebook.
# Override without editing code by setting environment variable DOCKML_BASE_DIR.
BASE_DIR = Path(
    os.environ.get("DOCKML_BASE_DIR", str(Path.cwd() / PROJECT_FOLDER))
).expanduser().resolve()
BASE_DIR.mkdir(parents=True, exist_ok=True)

DATA_CSV = BASE_DIR / "dataset.csv"
MODEL_PATH = BASE_DIR / "docking_score_model.joblib"
SCREENING_CSV = BASE_DIR / "screening_for_prediction.csv"
PREDICTIONS_CSV = BASE_DIR / "screening_docking_score_predictions.csv"
MORGAN_MAPPING_CSV = BASE_DIR / "morgan_bit_substructures.csv"
ENVIRONMENT_JSON = BASE_DIR / "environment_versions.json"

FIGURES_DIR = BASE_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## Biological target and docking metadata

This section automatically builds the scientific **identity card** of the model from **USER CONFIGURATION**.

The values document the disease, biological target, protein structure and docking protocol. They are saved with the trained artifact and Model Card but are **not molecular ML features**.

This provenance matters because a docking-score model should be interpreted within the same target and docking context represented by its training data.

In [ ]:
# Automatically generated model/campaign metadata.
# Edit USER CONFIGURATION, not this dictionary.

TARGET_METADATA = {
    "molflood_version": MOLFLOOD_VERSION,
    "molflood_release_status": MOLFLOOD_RELEASE_STATUS,
    "molflood_authors": MOLFLOOD_AUTHORS,

    "campaign_id": CAMPAIGN_ID,
    "campaign_title": CAMPAIGN_TITLE,

    "disease_name": DISEASE_NAME,
    "disease_category": DISEASE_CATEGORY,
    "disease_context": DISEASE_CONTEXT,

    "target_name": TARGET_NAME,
    "target_gene": TARGET_GENE,
    "target_uniprot_id": TARGET_UNIPROT_ID,
    "target_organism": TARGET_ORGANISM,
    "target_description": TARGET_DESCRIPTION,

    "structure_source": STRUCTURE_SOURCE,
    "pdb_id": PDB_ID,
    "protein_chain": PROTEIN_CHAIN,
    "structure_notes": STRUCTURE_NOTES,

    "docking_software": DOCKING_SOFTWARE,
    "docking_software_version": DOCKING_SOFTWARE_VERSION,
    "docking_protocol_id": DOCKING_PROTOCOL_ID,
    "docking_score_name": "docking_score",
    "docking_score_units": "kcal/mol",
    "docking_score_direction": "lower_is_more_favorable",

    "binding_site_definition": BINDING_SITE_DEFINITION,
    "grid_center": GRID_CENTER,
    "grid_size": GRID_SIZE,
    "exhaustiveness": EXHAUSTIVENESS,
    "ligand_preparation": LIGAND_PREPARATION,
    "protein_preparation": PROTEIN_PREPARATION,

    "project_scope": (
        "Machine-learning prioritization of molecular docking scores with emphasis "
        "on therapeutic targets associated with understudied or neglected diseases."
    ),
    "intended_use": (
        "Prioritization of molecules for subsequent computational evaluation "
        "under the same docking context."
    ),
    "out_of_scope": [
        "Experimental binding-affinity prediction",
        "Direct prediction of therapeutic efficacy",
        "Prediction for a different biological target without retraining",
        "Prediction for an incompatible docking protocol without validation",
    ],
}

display(pd.DataFrame(
    [(key, value) for key, value in TARGET_METADATA.items()],
    columns=["metadata_field", "value"],
))

In [ ]:
def validate_target_metadata(metadata_dict, strict=False):
    """
    Validate campaign metadata without affecting model training.

    strict=False:
        warns about fields that should be completed before public release.
    strict=True:
        raises an error when essential public metadata is missing.
    """
    essential_fields = [
        "campaign_id",
        "disease_name",
        "target_name",
        "target_organism",
        "docking_software",
        "docking_protocol_id",
        "docking_score_name",
        "docking_score_units",
    ]

    missing_or_placeholder = []
    for field in essential_fields:
        value = metadata_dict.get(field)
        if value is None or str(value).strip() == "" or "replace_with" in str(value):
            missing_or_placeholder.append(field)

    if missing_or_placeholder:
        message = (
            "Setup incomplete. Fill these fields in USER CONFIGURATION before public release: "
            + ", ".join(missing_or_placeholder)
        )
        if strict:
            raise ValueError(message)
        print("WARNING:", message)
    else:
        print("Target metadata validation: OK")

    if metadata_dict.get("docking_score_name") != "docking_score":
        raise ValueError(
            "This notebook is standardized to the predictive endpoint 'docking_score'."
        )

    return missing_or_placeholder


_ = validate_target_metadata(TARGET_METADATA, strict=PUBLICATION_MODE)

These two helper functions make file handling clearer: whenever an input file is required, the notebook shows exactly where it should be placed; whenever something is saved, the notebook prints the full output path.

In [ ]:
def require_file(path, description):
    """Check whether an input file exists and explain where to place it if it does not."""
    path = Path(path)
    if not path.exists():
        print("=" * 60)
        print(f"FILE NOT FOUND: {description}")
        print("=" * 60)
        print("Place the file exactly at this path and run the cell again:")
        print(f"  {path.resolve()}")
        print("=" * 60)
        raise FileNotFoundError(f"Expected file not found: {path.resolve()}")

    print(f"Reading {description} de:\n  {path.resolve()}")
    return path


def announce_saved(path, description):
    """Clearly report where a file was saved on the computer."""
    path = Path(path)
    print("=" * 60)
    print(f"{description.upper()} SAVED SUCCESSFULLY")
    print("=" * 60)
    print(f"Full path: {path.resolve()}")
    print("=" * 60)

def show_expected_training_csv():
    print("Expected CSV: smiles,docking_score")
    print("Example: CCO,-5.8")


## 3. Dataset loading

The input CSV must contain at least:

- `smiles`
- `docking_score`

The dataset is assumed to have been **deduplicated externally before this notebook**. No chemical deduplication is performed here.

### What you need to do here

Place `dataset.csv` in the project folder. Required columns are `smiles` and `docking_score`. The dataset is expected to have been deduplicated before this notebook.

| column | meaning |
|---|---|
| `smiles` | molecular structure |
| `docking_score` | score generated by the documented docking campaign |

### Tutorial dataset

A public MolFlood release may include a developer-provided tutorial dataset. Running that dataset first is recommended to verify the environment and learn the outputs before substituting a new docking campaign.

The production input remains `dataset.csv` containing at least `smiles` and `docking_score`.

In [ ]:
if not DATA_CSV.exists():
    show_expected_training_csv()

df = pd.read_csv(
    require_file(DATA_CSV, "training dataset (columns: smiles, docking_score)")
)
df.columns = df.columns.str.strip()
df.head()

In [ ]:
required_cols = {"smiles", "docking_score"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(
        f"Missing required columns in the CSV: {sorted(missing)}. "
        "Expected target column name: 'docking_score'."
    )

df = df[["smiles", "docking_score"]].copy()
df["smiles"] = df["smiles"].astype(str).str.strip()
df["docking_score"] = pd.to_numeric(df["docking_score"], errors="coerce")
df["docking_score"] = df["docking_score"].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["smiles", "docking_score"]).reset_index(drop=True)

print("Samples after schema/missing-value cleaning:", len(df))
print("Chemical deduplication: expected to have been performed externally.")
df["docking_score"].describe()

In [ ]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Number of molecules: {len(df)}")
print(f"Available columns: {list(df.columns)}")
print(f"Unique input SMILES strings: {df['smiles'].nunique()}")
print(f"Minimum docking score: {df['docking_score'].min():.3f}")
print(f"Maximum docking score: {df['docking_score'].max():.3f}")
print(f"Mean docking score: {df['docking_score'].mean():.3f}")
print(f"Docking score standard deviation: {df['docking_score'].std():.3f}")
print("=" * 60)
display(df.head())

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df["docking_score"], kde=True, bins=30)
plt.xlabel("Docking score")
plt.ylabel("Frequency")
plt.title("Distribution of docking score values")
plt.show()

This pipeline uses Morgan fingerprints as the main structural representation and enriches them with RDKit physicochemical descriptors computed from SMILES. Morgan fingerprints remain the default because they are strong, scalable, and widely used for QSAR-like tabular molecular modeling. Additional fingerprint families were intentionally removed from the current workflow to reduce dimensionality, noise, and unnecessary computational cost. Future versions of MolFlood may incorporate and systematically evaluate additional molecular fingerprint representations, enabling broader comparisons of their impact on predictive performance, generalization, and computational efficiency.

This pipeline uses Morgan fingerprints as the main structural representation and enriches them with RDKit physicochemical descriptors computed from SMILES. Morgan fingerprints remain the default because they are strong, scalable and widely used for QSAR-like tabular molecular modeling. Additional fingerprint families were intentionally removed from the main workflow to reduce dimensionality, noise and unnecessary computational cost.

### Fingerprint configuration and held-out test protection

For publication, the Morgan fingerprint configuration must be specified **a priori** or selected using training-only cross-validation. Do not compare multiple fingerprint presets using the held-out test set and then retain the best one; that would make the test set part of model selection.

### 4.1 Morgan presets

The Morgan representation is selected in the Colab control above (`FINGERPRINT_PRESET`). Recommended presets:

- `morgan_2048_r2`: default ECFP4-like representation; recommended starting point.
- `morgan_1024_r2`: lighter and faster; useful for quick tests.
- `morgan_4096_r2`: less hashing collision, but more features.
- `morgan_2048_r3`: larger local neighborhoods; may help if broader substructures matter.

Keep the final choice empirical: compare RMSE, MAE, R2, Spearman and top-k ranking metrics.

In [ ]:
CONTINUOUS_COLS = [
    "MolWt", "ExactMolWt", "MolLogP", "TPSA", "MolMR",
    "NumHDonors", "NumHAcceptors", "NumRotatableBonds",
    "HeavyAtomCount", "NumHeteroatoms", "NOCount", "NHOHCount",
    "RingCount", "NumAromaticRings", "NumAliphaticRings", "NumSaturatedRings",
    "FractionCSP3", "BertzCT", "LabuteASA", "BalabanJ",
    "FormalCharge", "NumValenceElectrons", "MaxPartialCharge", "MinPartialCharge",
]

FINGERPRINT_EXPERIMENTS = {
    "morgan_1024_r2": {"morgan_radius": 2, "morgan_bits": 1024},
    "morgan_2048_r2": {"morgan_radius": 2, "morgan_bits": 2048},
    "morgan_4096_r2": {"morgan_radius": 2, "morgan_bits": 4096},
    "morgan_2048_r3": {"morgan_radius": 3, "morgan_bits": 2048},
}

#@title Morgan fingerprint settings
FINGERPRINT_PRESET = "morgan_2048_r2" #@param ["morgan_1024_r2", "morgan_2048_r2", "morgan_4096_r2", "morgan_2048_r3"]

FINGERPRINT_CONFIG = FINGERPRINT_EXPERIMENTS[FINGERPRINT_PRESET]

print("=" * 60)
print("SELECTED MORGAN FINGERPRINT CONFIGURATION")
print("=" * 60)
print(FINGERPRINT_CONFIG)
print("=" * 60)


def safe_descriptor(func, mol, default=np.nan):
    try:
        value = func(mol)
        if value is None or not np.isfinite(value):
            return default
        return value
    except Exception:
        return default


def calculate_partial_charges(mol):
    mol_h = Chem.AddHs(mol)
    try:
        AllChem.ComputeGasteigerCharges(mol_h)
        charges = []
        for atom in mol_h.GetAtoms():
            charge = atom.GetProp("_GasteigerCharge")
            if charge not in ["nan", "-nan", "inf", "-inf"]:
                charges.append(float(charge))
        if not charges:
            return np.nan, np.nan
        return max(charges), min(charges)
    except Exception:
        return np.nan, np.nan


def bitvect_to_array(bitvect, n_bits):
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(bitvect, arr)
    return arr


def add_morgan_features(features, mol, fp_config):
    n_bits = fp_config.get("morgan_bits", 2048)
    radius = fp_config.get("morgan_radius", 2)

    morgan_fp = AllChem.GetMorganFingerprintAsBitVect(
        mol,
        radius,
        nBits=n_bits
    )
    arr = bitvect_to_array(morgan_fp, n_bits=n_bits)

    for i, value in enumerate(arr):
        features[f"MORGAN_{i}"] = int(value)


def calculate_features(smiles, fp_config=FINGERPRINT_CONFIG):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    max_partial_charge, min_partial_charge = calculate_partial_charges(mol)

    features = {
        "MolWt": Descriptors.MolWt(mol),
        "ExactMolWt": Descriptors.ExactMolWt(mol),
        "MolLogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "MolMR": Descriptors.MolMR(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "NumHeteroatoms": Descriptors.NumHeteroatoms(mol),
        "NOCount": Descriptors.NOCount(mol),
        "NHOHCount": Descriptors.NHOHCount(mol),
        "RingCount": Descriptors.RingCount(mol),
        "NumAromaticRings": Descriptors.NumAromaticRings(mol),
        "NumAliphaticRings": Descriptors.NumAliphaticRings(mol),
        "NumSaturatedRings": Descriptors.NumSaturatedRings(mol),
        "FractionCSP3": Descriptors.FractionCSP3(mol),
        "BertzCT": Descriptors.BertzCT(mol),
        "LabuteASA": safe_descriptor(Descriptors.LabuteASA, mol),
        "BalabanJ": safe_descriptor(Descriptors.BalabanJ, mol),
        "FormalCharge": Chem.GetFormalCharge(mol),
        "NumValenceElectrons": Descriptors.NumValenceElectrons(mol),
        "MaxPartialCharge": max_partial_charge,
        "MinPartialCharge": min_partial_charge,
    }

    add_morgan_features(features, mol, fp_config)
    return features


def build_feature_table(dataframe, smiles_col="smiles", fp_config=FINGERPRINT_CONFIG):
    rows = []
    valid_idx = []

    for idx, smiles in dataframe[smiles_col].items():
        feats = calculate_features(smiles, fp_config=fp_config)
        if feats is not None:
            rows.append(feats)
            valid_idx.append(idx)

    X = pd.DataFrame(rows)
    valid_df = dataframe.loc[valid_idx].reset_index(drop=True)
    X = X.reset_index(drop=True)
    return X, valid_df


X, df_valid = build_feature_table(df, fp_config=FINGERPRINT_CONFIG)
y = df_valid["docking_score"].copy()

print("Valid molecules:", len(df_valid))
print("Number of features:", X.shape[1])
X.head()

In [ ]:
# Memory diagnostic for the dense molecular feature matrix.
feature_memory_mb = float(X.memory_usage(index=True, deep=True).sum() / (1024 ** 2))
print(f"Feature matrix memory: {feature_memory_mb:.1f} MB")

if feature_memory_mb > 1500:
    warnings.warn(
        "The dense Morgan + descriptor matrix exceeds ~1.5 GB. "
        "For very large public datasets, consider a sparse-feature backend "
        "before increasing dataset size further."
    )

In [ ]:
invalid_count = len(df) - len(df_valid)

print("=" * 60)
print("FEATURE SUMMARY")
print("=" * 60)
print(f"Original molecules: {len(df)}")
print(f"RDKit-valid molecules: {len(df_valid)}")
print(f"Molecules removed due to invalid SMILES: {invalid_count}")
print(f"Physicochemical descriptors: {len(CONTINUOUS_COLS)}")
print(f"Morgan fingerprint configuration: {FINGERPRINT_CONFIG}")
print(f"Total generated features: {X.shape[1]}")
print("=" * 60)

display(X[CONTINUOUS_COLS].describe().T)

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(X[CONTINUOUS_COLS].corr(), cmap="coolwarm", center=0)
plt.title("Correlation among physicochemical descriptors")
plt.show()

In [ ]:
fingerprint_prefixes = ("MORGAN_",)
fingerprint_cols = [col for col in X.columns if col.startswith(fingerprint_prefixes)]
fp_activation = X[fingerprint_cols].mean().sort_values(ascending=False)

print("Top 10 most frequent Morgan bits:")
display(fp_activation.head(10).to_frame("frequency"))

print("Top 10 least frequent Morgan bits:")
display(fp_activation.tail(10).to_frame("frequency"))

### In plain language: why the data are split

Part of the molecules is kept as a held-out test set and is not used to choose the algorithm or tune hyperparameters. Structural grouping makes evaluation more demanding by reducing close-analogue overlap across fitting and validation/test partitions.

## 5. Train/test split and structural validation

The test set is a **final hold-out set**. It is created here and is not used for baseline comparison, algorithm selection or hyperparameter optimization.

- `random_stratified`: approximately preserves the docking-score distribution across train/test. Training CV uses seeded shuffled `KFold`.
- `scaffold`: separates Bemis-Murcko scaffold groups. Training CV uses `GroupKFold`.
- `morgan_kmeans_cluster`: separates Morgan-fingerprint clusters. Training CV uses `GroupKFold`.

For structural validation, a structural group can never appear simultaneously in a fitting fold and its validation fold.

In [ ]:
#@title Train/test split settings

SPLIT_MODE_MAP = {
    "random_stratified": "random_stratified",
    "scaffold": "scaffold",
    "fingerprint_cluster": "morgan_kmeans_cluster",
}

if SPLIT_MODE_USER not in SPLIT_MODE_MAP:
    raise ValueError(
        f"Unknown SPLIT_MODE_USER={SPLIT_MODE_USER!r}. "
        f"Choose one of {list(SPLIT_MODE_MAP)}."
    )

SPLIT_MODE = SPLIT_MODE_MAP[SPLIT_MODE_USER]

TEST_SIZE = 0.2 #@param {type:"slider", min:0.1, max:0.3, step:0.05}
N_STRUCTURAL_CLUSTERS = 100 #@param {type:"integer"}

fingerprint_prefixes = ("MORGAN_",)
fp_cols = [col for col in X.columns if col.startswith(fingerprint_prefixes)]

def make_docking_score_bins(y_values, q=10, min_bin_size=2):
    """
    Build quantile bins suitable for stratified train/test splitting.

    The number of bins is reduced automatically until every bin has at least
    `min_bin_size` observations. If this is impossible, return None and let
    the caller use a seeded unstratified split rather than fail unexpectedly.
    """
    y_series = pd.Series(y_values).reset_index(drop=True)
    max_q = min(int(q), max(2, len(y_series) // min_bin_size))

    for current_q in range(max_q, 1, -1):
        try:
            bins = pd.qcut(y_series, q=current_q, duplicates="drop")
            counts = bins.value_counts()
            if len(counts) >= 2 and counts.min() >= min_bin_size:
                return pd.Series(bins.values, index=pd.Index(y_values.index))
        except ValueError:
            continue

    return None

def scaffold_key(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "INVALID"
    scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    if scaffold:
        return scaffold
    return Chem.MolToSmiles(mol, canonical=True)

def grouped_train_test_split(X_data, y_data, groups, test_size=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    groups_series = pd.Series(np.asarray(groups), index=X_data.index)
    group_to_indices = {
        group: group_df.index.tolist()
        for group, group_df in groups_series.groupby(groups_series)
    }

    group_items = list(group_to_indices.items())
    rng.shuffle(group_items)
    group_items = sorted(group_items, key=lambda item: len(item[1]), reverse=True)

    n_target_test = int(round(len(X_data) * test_size))
    test_indices, train_indices = [], []

    for _, indices in group_items:
        if len(test_indices) < n_target_test:
            test_indices.extend(indices)
        else:
            train_indices.extend(indices)

    train_indices = sorted(train_indices)
    test_indices = sorted(test_indices)

    return (
        X_data.loc[train_indices],
        X_data.loc[test_indices],
        y_data.loc[train_indices],
        y_data.loc[test_indices],
    )

def morgan_kmeans_cluster_labels(X_data, fp_columns, n_clusters=100, random_state=42):
    n_clusters = min(n_clusters, max(2, len(X_data) // 20))
    X_fp = X_data[fp_columns].astype(np.float32)

    clusterer = MiniBatchKMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        batch_size=4096,
        n_init=10,
        verbose=0,
    )
    return clusterer.fit_predict(X_fp)

if SPLIT_MODE == "random_stratified":
    docking_score_bins = make_docking_score_bins(y)
    if docking_score_bins is None:
        warnings.warn(
            "Could not construct stable docking-score strata; "
            "falling back to a seeded unstratified train/test split.",
            RuntimeWarning,
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=docking_score_bins,
    )
    split_groups_series = None
    train_groups = None

elif SPLIT_MODE == "scaffold":
    split_groups_series = pd.Series(
        df_valid["smiles"].apply(scaffold_key).values,
        index=X.index,
        name="structural_group",
    )
    X_train, X_test, y_train, y_test = grouped_train_test_split(
        X, y, split_groups_series.values, TEST_SIZE, RANDOM_STATE
    )
    train_groups = split_groups_series.loc[X_train.index]

elif SPLIT_MODE == "morgan_kmeans_cluster":
    split_groups_series = pd.Series(
        morgan_kmeans_cluster_labels(
            X, fp_cols, N_STRUCTURAL_CLUSTERS, RANDOM_STATE
        ),
        index=X.index,
        name="structural_group",
    )
    X_train, X_test, y_train, y_test = grouped_train_test_split(
        X, y, split_groups_series.values, TEST_SIZE, RANDOM_STATE
    )
    train_groups = split_groups_series.loc[X_train.index]

else:
    raise ValueError(f"Unknown SPLIT_MODE: {SPLIT_MODE}")

def make_training_cv(split_mode, n_splits, train_groups=None):
    if split_mode == "random_stratified":
        return KFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=RANDOM_STATE,
        ), None

    if train_groups is None:
        raise ValueError("Structural CV requires training group labels.")

    n_unique_groups = pd.Series(train_groups).nunique()
    if n_unique_groups < n_splits:
        raise ValueError(
            f"GroupKFold requires at least {n_splits} unique training groups; "
            f"only {n_unique_groups} were found."
        )

    return GroupKFold(n_splits=n_splits), train_groups

if len(X_train) == 0 or len(X_test) == 0:
    raise ValueError(
        "The selected split produced an empty train or test set. "
        "Use more molecules, fewer/larger structural groups, or adjust TEST_SIZE."
    )

cv_strategy, cv_groups_train = make_training_cv(
    SPLIT_MODE,
    CV_FOLDS,
    train_groups=train_groups,
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("CV strategy:", type(cv_strategy).__name__)

In [ ]:
print("=" * 60)
print("TRAIN/TEST SPLIT")
print("=" * 60)
print(f"Split mode: {SPLIT_MODE}")
print(f"Train: {X_train.shape[0]} molecules ({100 * len(X_train) / len(X):.1f}%)")
print(f"Test: {X_test.shape[0]} molecules ({100 * len(X_test) / len(X):.1f}%)")
print(f"Features per molecule: {X_train.shape[1]}")
print(f"Mean training docking score: {y_train.mean():.3f}")
print(f"Mean test docking score: {y_test.mean():.3f}")
print(f"Training CV: {type(cv_strategy).__name__} ({CV_FOLDS} folds)")

if split_groups_series is not None:
    train_group_set = set(split_groups_series.loc[X_train.index])
    test_group_set = set(split_groups_series.loc[X_test.index])
    overlap = train_group_set.intersection(test_group_set)
    print(f"Number of structural groups: {split_groups_series.nunique()}")
    print(f"Train/test structural-group overlap: {len(overlap)}")
    assert len(overlap) == 0, "Structural leakage detected between train and test."

print("=" * 60)

In [ ]:
target_distribution_summary = pd.DataFrame({
    "train": y_train.describe(),
    "test": y_test.describe(),
})
print("Docking-score distribution by partition:")
display(target_distribution_summary)

mean_shift = float(abs(y_train.mean() - y_test.mean()))
pooled_std = float(pd.concat([y_train, y_test]).std())

if pooled_std > 0:
    standardized_mean_shift = mean_shift / pooled_std
    print(f"Standardized train/test mean shift: {standardized_mean_shift:.3f}")
    if standardized_mean_shift > 0.5:
        warnings.warn(
            "Train and test docking-score means differ substantially. "
            "This may reflect a genuine structural/domain shift and should be "
            "reported when interpreting held-out performance."
        )

### Cross-validation fold diagnostics

Group-aware CV prevents structural-group leakage, but it does not guarantee identical docking-score distributions across folds. These diagnostics make fold size, target distribution and structural-group separation explicit.

In [ ]:
def build_cv_fold_diagnostics(X_data, y_data, cv, groups=None):
    Xr = X_data.reset_index(drop=True)
    yr = y_data.reset_index(drop=True)
    gv = None if groups is None else np.asarray(groups)
    rows = []

    for fold, (fit_idx, val_idx) in enumerate(cv.split(Xr, yr, gv), start=1):
        row = {
            "fold": fold,
            "n_train": len(fit_idx),
            "n_validation": len(val_idx),
            "train_target_mean": yr.iloc[fit_idx].mean(),
            "train_target_std": yr.iloc[fit_idx].std(),
            "validation_target_mean": yr.iloc[val_idx].mean(),
            "validation_target_std": yr.iloc[val_idx].std(),
            "validation_target_min": yr.iloc[val_idx].min(),
            "validation_target_max": yr.iloc[val_idx].max(),
        }
        if gv is not None:
            train_groups_fold = set(gv[fit_idx])
            val_groups_fold = set(gv[val_idx])
            row["n_train_groups"] = len(train_groups_fold)
            row["n_validation_groups"] = len(val_groups_fold)
            row["group_overlap"] = len(train_groups_fold.intersection(val_groups_fold))
        rows.append(row)

    return pd.DataFrame(rows)

cv_fold_diagnostics = build_cv_fold_diagnostics(
    X_train, y_train, cv_strategy, cv_groups_train
)
display(cv_fold_diagnostics)

if "group_overlap" in cv_fold_diagnostics.columns:
    assert (cv_fold_diagnostics["group_overlap"] == 0).all()

## 6. Preprocessing

`VarianceThreshold` removes fingerprint bits with no variance or very low variance. It is included inside the pipeline so it is fitted only on the training set.

In [ ]:
def make_preprocessor(variance_threshold=0.0):
    continuous_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    fingerprint_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("variance", VarianceThreshold(threshold=variance_threshold)),
    ])

    return ColumnTransformer([
        ("continuous", continuous_pipe, CONTINUOUS_COLS),
        ("fingerprints", fingerprint_pipe, fp_cols),
    ])

## 7. Algorithm selection using training-only cross-validation

The held-out test set is intentionally absent from this section.

Each candidate algorithm receives out-of-fold predictions on `X_train` only. The algorithm with the lowest out-of-fold RMSE is selected for Optuna. Under structural split modes, this selection also uses `GroupKFold`.

Only after algorithm and hyperparameter selection is complete is the model evaluated on `X_test`.

In [ ]:
def regression_metrics(y_true, y_pred):
    spearman = spearmanr(y_true, y_pred).correlation
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Spearman": spearman,
    }

def top_k_ranking_metrics(y_true, y_pred, fractions=(0.01, 0.05, 0.10)):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    n = len(y_true)
    rows = []

    for frac in fractions:
        k = max(1, int(round(n * frac)))
        true_top = set(y_true.nsmallest(k).index)
        pred_top = set(y_pred.nsmallest(k).index)
        overlap = len(true_top.intersection(pred_top))

        rows.append({
            "top_fraction": frac,
            "k": k,
            "top_k_overlap": overlap,
            "precision_at_k": overlap / k,
            "recall_at_k": overlap / k,
            "mean_true_docking_score_in_pred_top_k": y_true.loc[list(pred_top)].mean(),
            "best_true_docking_score_in_pred_top_k": y_true.loc[list(pred_top)].min(),
        })

    return pd.DataFrame(rows)

baseline_models = {
    "DummyMean": DummyRegressor(strategy="mean"),
    "RandomForest": RandomForestRegressor(
        n_estimators=500, random_state=RANDOM_STATE, n_jobs=MODEL_N_JOBS
    ),
    "ExtraTrees": ExtraTreesRegressor(
        n_estimators=500, random_state=RANDOM_STATE, n_jobs=MODEL_N_JOBS
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=MODEL_N_JOBS,
    ),
}

if LGBMRegressor is not None:
    baseline_models["LightGBM"] = LGBMRegressor(
        n_estimators=800,
        learning_rate=0.03,
        num_leaves=64,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=MODEL_N_JOBS,
        deterministic=REPRODUCIBLE_MODE,
        force_col_wise=True,
        verbose=-1,
    )

if CatBoostRegressor is not None:
    baseline_models["CatBoost"] = CatBoostRegressor(
        iterations=800,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        thread_count=MODEL_N_JOBS,
        verbose=False,
        allow_writing_files=False,
    )

baseline_results = {}
baseline_oof_predictions = {}

for name, model in baseline_models.items():
    pipe = Pipeline([
        ("preprocess", make_preprocessor(variance_threshold=0.0)),
        ("model", model),
    ])

    oof_pred = cross_val_predict(
        pipe,
        X_train,
        y_train,
        cv=cv_strategy,
        groups=cv_groups_train,
        n_jobs=CV_N_JOBS,
        method="predict",
    )
    baseline_oof_predictions[name] = oof_pred
    baseline_results[name] = regression_metrics(y_train, oof_pred)

baseline_df = pd.DataFrame(baseline_results).T.sort_values("RMSE")
display(baseline_df)

In [ ]:
print("=" * 60)
print("ALGORITHM SELECTION — TRAINING-ONLY CROSS-VALIDATION")
print("=" * 60)
display(baseline_df)

eligible_model_names = [name for name in baseline_df.index if name != "DummyMean"]
best_baseline_name = baseline_df.loc[eligible_model_names].sort_values("RMSE").index[0]

dummy_rmse = baseline_df.loc["DummyMean", "RMSE"]
best_model_rmse = baseline_df.loc[best_baseline_name, "RMSE"]
relative_rmse_improvement_vs_dummy = (dummy_rmse - best_model_rmse) / dummy_rmse

print(
    f"Best candidate improves CV RMSE over DummyMean by "
    f"{relative_rmse_improvement_vs_dummy:.1%}"
)
print(f"Selected algorithm by out-of-fold training RMSE: {best_baseline_name}")
print("Held-out test set used in selection: NO")

In [ ]:
# Publication-ready baseline plots
MODEL_COLORS = {
    "DummyMean": "#7f7f7f",
    "RandomForest": "#1f77b4",
    "ExtraTrees": "#ff7f0e",
    "XGBoost": "#2ca02c",
    "LightGBM": "#d62728",
    "CatBoost": "#9467bd",
}

MODEL_LABELS = {
    "DummyMean": "Dummy mean",
    "RandomForest": "Random Forest",
    "ExtraTrees": "Extra Trees",
    "XGBoost": "XGBoost",
    "LightGBM": "LightGBM",
    "CatBoost": "CatBoost",
}


def set_publication_style():
    sns.set_theme(style="whitegrid", context="talk")
    plt.rcParams.update({
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.family": "DejaVu Sans",
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 11,
        "axes.edgecolor": "#333333",
        "axes.linewidth": 1.0,
        "grid.color": "#D0D0D0",
        "grid.linewidth": 0.8,
    })


def save_figure(fig, output_stem):
    png_path = FIGURES_DIR / f"{output_stem}.png"
    pdf_path = FIGURES_DIR / f"{output_stem}.pdf"
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


def polish_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(True, axis="y", alpha=0.35)
    return ax

def plot_grouped_metric_bars(metrics_df, metrics, title, ylabel, output_stem):
    set_publication_style()

    plot_df = (
        metrics_df[metrics]
        .reset_index()
        .rename(columns={"index": "Model"})
        .melt(id_vars="Model", var_name="Metric", value_name="Value")
    )
    plot_df["ModelLabel"] = plot_df["Model"].map(MODEL_LABELS).fillna(plot_df["Model"])

    fig, ax = plt.subplots(figsize=(11, 6.5))

    sns.barplot(
        data=plot_df,
        x="Metric",
        y="Value",
        hue="ModelLabel",
        palette={MODEL_LABELS.get(k, k): v for k, v in MODEL_COLORS.items()},
        edgecolor="black",
        linewidth=0.6,
        ax=ax,
    )

    ax.set_title(title, pad=14, weight="bold")
    ax.set_xlabel("Metric")
    ax.set_ylabel(ylabel)
    ax.legend(title="Algorithm", frameon=True, loc="best")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", fontsize=8, padding=2, rotation=90)

    fig.tight_layout()

    png_path = FIGURES_DIR / f"{output_stem}.png"
    pdf_path = FIGURES_DIR / f"{output_stem}.pdf"
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")
    plt.show()


plot_grouped_metric_bars(
    baseline_df,
    metrics=["RMSE", "MAE"],
    title="Training-CV Regression Error Across Algorithms",
    ylabel="Error (kcal/mol)",
    output_stem="training_cv_error_metrics",
)

plot_grouped_metric_bars(
    baseline_df,
    metrics=["R2", "Spearman"],
    title="Training-CV Predictive Performance Across Algorithms",
    ylabel="Score",
    output_stem="training_cv_score_metrics",
)

### In plain language: what Optuna and GroupKFold do

Optuna tests model settings using only training data. With a structural split, `GroupKFold` moves whole scaffold/cluster groups between folds, so the same structural group cannot appear on both fitting and validation sides of one fold.

### Publication workload summary

MolFlood keeps the full publication-grade validation workload enabled. Large datasets may therefore require substantial computation. The next cell reports the main workload before hyperparameter optimization begins.

Estimator-level parallelism is enabled, while cross-validation stays single-level to avoid nested CPU oversubscription.

In [ ]:
approx_optuna_fits = OPTUNA_TRIALS * CV_FOLDS
approx_y_scrambling_fits = Y_SCRAMBLING_REPEATS * CV_FOLDS

print("=" * 72)
print("PUBLICATION WORKLOAD")
print("=" * 72)
print(f"Training molecules: {len(X_train):,}")
print(f"Features: {X_train.shape[1]:,}")
print(f"CV folds: {CV_FOLDS}")
print(f"Optuna trials: {OPTUNA_TRIALS}")
print(f"Approx. Optuna model fits: {approx_optuna_fits:,}")
print(f"Bootstrap resamples: {BOOTSTRAP_REPEATS:,}")
print(f"Y-scrambling repetitions: {Y_SCRAMBLING_REPEATS}")
print(f"Approx. Y-scrambling model fits: {approx_y_scrambling_fits:,}")
print(f"Estimator jobs: {MODEL_N_JOBS}")
print(f"CV jobs: {CV_N_JOBS}")
print("=" * 72)

## 8. Optuna with leakage-resistant cross-validation

Optuna sees only `X_train` and `y_train`.

### How `GroupKFold` works here

With a structural split, every training molecule has a structural group label: either a scaffold or a fingerprint-cluster ID. `GroupKFold` partitions **groups**, not individual rows.

For each Optuna trial:

1. a subset of structural groups is reserved as the validation fold;
2. all remaining groups form the fitting fold;
3. the same group can never be in fitting and validation simultaneously;
4. preprocessing is fitted only within the fitting fold because it remains inside the scikit-learn `Pipeline`;
5. Optuna minimizes the mean validation RMSE across folds.

This makes hyperparameter selection substantially less permissive than ordinary random KFold when structurally related molecules exist.

In [ ]:
#@title Optuna settings
OPTUNA_TRIALS = int(OPTUNA_TRIALS)
VARIANCE_THRESHOLD_MAX = 0.02 #@param {type:"number"}
FINAL_MODEL_TYPE_OVERRIDE = None

MODEL_NAME_TO_TYPE = {
    "RandomForest": "random_forest",
    "ExtraTrees": "extra_trees",
    "XGBoost": "xgboost",
    "LightGBM": "lightgbm",
    "CatBoost": "catboost",
}

FINAL_MODEL_TYPE = (
    FINAL_MODEL_TYPE_OVERRIDE
    if FINAL_MODEL_TYPE_OVERRIDE is not None
    else MODEL_NAME_TO_TYPE[best_baseline_name]
)

def build_trial_model(trial, model_type):
    if model_type == "random_forest":
        return RandomForestRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 1500),
            max_depth=trial.suggest_int("max_depth", 4, 40),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            max_features=trial.suggest_float("max_features", 0.2, 1.0),
            random_state=RANDOM_STATE,
            n_jobs=MODEL_N_JOBS,
        )

    if model_type == "extra_trees":
        return ExtraTreesRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 1500),
            max_depth=trial.suggest_int("max_depth", 4, 40),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
            max_features=trial.suggest_float("max_features", 0.2, 1.0),
            random_state=RANDOM_STATE,
            n_jobs=MODEL_N_JOBS,
        )

    if model_type == "xgboost":
        return XGBRegressor(
            n_estimators=trial.suggest_int("n_estimators", 200, 1200),
            max_depth=trial.suggest_int("max_depth", 2, 10),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            min_child_weight=trial.suggest_float("min_child_weight", 1e-2, 20.0, log=True),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
            gamma=trial.suggest_float("gamma", 1e-8, 10.0, log=True),
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=MODEL_N_JOBS,
        )

    if model_type == "lightgbm":
        if LGBMRegressor is None:
            raise ImportError("LightGBM is not installed.")
        return LGBMRegressor(
            n_estimators=trial.suggest_int("n_estimators", 300, 2000),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            num_leaves=trial.suggest_int("num_leaves", 16, 256),
            max_depth=trial.suggest_int("max_depth", -1, 16),
            min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
            subsample=trial.suggest_float("subsample", 0.6, 1.0),
            subsample_freq=1,
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 20.0, log=True),
            random_state=RANDOM_STATE,
            n_jobs=MODEL_N_JOBS,
            deterministic=REPRODUCIBLE_MODE,
            force_col_wise=True,
            verbose=-1,
        )

    if model_type == "catboost":
        if CatBoostRegressor is None:
            raise ImportError("CatBoost is not installed.")
        return CatBoostRegressor(
            iterations=trial.suggest_int("iterations", 300, 2000),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
            depth=trial.suggest_int("depth", 4, 10),
            l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1e-3, 20.0, log=True),
            random_strength=trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            thread_count=MODEL_N_JOBS,
            verbose=False,
            allow_writing_files=False,
        )

    raise ValueError(f"Unknown model_type: {model_type}")

def objective(trial):
    variance_threshold = trial.suggest_float(
        "variance_threshold", 0.0, VARIANCE_THRESHOLD_MAX
    )

    pipe = Pipeline([
        ("preprocess", make_preprocessor(variance_threshold=variance_threshold)),
        ("model", build_trial_model(trial, FINAL_MODEL_TYPE)),
    ])

    scores = cross_val_score(
        pipe,
        X_train,
        y_train,
        scoring="neg_root_mean_squared_error",
        cv=cv_strategy,
        groups=cv_groups_train,
        n_jobs=CV_N_JOBS,
    )
    return -scores.mean()

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

print("Selected final model type:", FINAL_MODEL_TYPE)
print("Best CV RMSE:", study.best_value)
print("Best parameters:")
display(study.best_params)

In [ ]:
trials_df = study.trials_dataframe()

print("=" * 60)
print("OPTUNA OPTIMIZATION SUMMARY")
print("=" * 60)
print(f"Selected model: {FINAL_MODEL_TYPE}")
print(f"Number of trials: {len(trials_df)}")
print(f"Best mean CV RMSE: {study.best_value:.4f}")
print("Best hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print("=" * 60)

display(trials_df.sort_values("value").head(10))

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(trials_df["number"], trials_df["value"], marker="o", linewidth=1)
plt.axhline(study.best_value, color="red", linestyle="--", label=f"Best RMSE: {study.best_value:.3f}")
plt.xlabel("Tentativa Optuna")
plt.ylabel("Média CV RMSE")
plt.title(f"Histórico de otimização - {FINAL_MODEL_TYPE}")
plt.legend()
plt.show()

## 9. Final training and test evaluation

In [ ]:
best_params = study.best_params.copy()
best_variance_threshold = best_params.pop("variance_threshold")

fixed_trial = optuna.trial.FixedTrial(best_params)
final_estimator = build_trial_model(fixed_trial, FINAL_MODEL_TYPE)

final_model = Pipeline([
    ("preprocess", make_preprocessor(variance_threshold=best_variance_threshold)),
    ("model", final_estimator),
])

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

test_metrics = regression_metrics(y_test, y_pred)
test_metrics

### 9.1 Model fit diagnostics: train vs test

This section checks whether the model may be overfitting or underfitting. A large train-test gap suggests overfitting. High errors in both train and test suggest underfitting or insufficient molecular representation.

In [ ]:
train_pred = final_model.predict(X_train)

train_metrics = regression_metrics(y_train, train_pred)
test_metrics = regression_metrics(y_test, y_pred)

fit_diagnostics_df = pd.DataFrame({
    "Train": train_metrics,
    "Test": test_metrics,
})

fit_diagnostics_df["Generalization_gap"] = pd.Series({
    "RMSE": test_metrics["RMSE"] - train_metrics["RMSE"],
    "MAE": test_metrics["MAE"] - train_metrics["MAE"],
    "R2": train_metrics["R2"] - test_metrics["R2"],
    "Spearman": train_metrics["Spearman"] - test_metrics["Spearman"],
})

display(fit_diagnostics_df)

In [ ]:
gap_interpretation = {
    "RMSE_gap_test_minus_train": test_metrics["RMSE"] - train_metrics["RMSE"],
    "MAE_gap_test_minus_train": test_metrics["MAE"] - train_metrics["MAE"],
    "R2_gap_train_minus_test": train_metrics["R2"] - test_metrics["R2"],
    "Spearman_gap_train_minus_test": train_metrics["Spearman"] - test_metrics["Spearman"],
}

print("=" * 60)
print("GENERALIZATION GAP")
print("=" * 60)
for key, value in gap_interpretation.items():
    print(f"{key}: {value:.4f}")
print("=" * 60)

In [ ]:
fit_diagnostics_df.loc[["RMSE", "MAE"]][["Train", "Test"]].plot(
    kind="bar",
    figsize=(8, 5),
    color=["#4C78A8", "#F58518"],
    edgecolor="black"
)
plt.ylabel("Error")
plt.title("Train vs test error")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

fit_diagnostics_df.loc[["R2", "Spearman"]][["Train", "Test"]].plot(
    kind="bar",
    figsize=(8, 5),
    color=["#4C78A8", "#F58518"],
    edgecolor="black"
)
plt.ylabel("Score")
plt.title("Train vs test ranking/explained-variance metrics")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Optimized model: out-of-fold training performance

After hyperparameter optimization, the frozen pipeline is evaluated again using the same training-only CV strategy. Reporting every fold exposes instability that would be hidden by a single average score.

In [ ]:
def metric_row(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Spearman": spearmanr(y_true, y_pred).correlation,
    }

final_cv_rows = []
final_oof_pred = np.empty(len(X_train), dtype=float)
groups_array = None if cv_groups_train is None else np.asarray(cv_groups_train)

for fold, (fit_idx, val_idx) in enumerate(
    cv_strategy.split(X_train, y_train, groups_array),
    start=1,
):
    fold_model = clone(final_model)
    fold_model.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
    fold_pred = fold_model.predict(X_train.iloc[val_idx])
    final_oof_pred[val_idx] = fold_pred

    row = {"fold": fold, **metric_row(y_train.iloc[val_idx], fold_pred)}
    row["n_validation"] = len(val_idx)
    if groups_array is not None:
        row["n_validation_groups"] = len(set(groups_array[val_idx]))
    final_cv_rows.append(row)

final_cv_metrics_df = pd.DataFrame(final_cv_rows)
display(final_cv_metrics_df)

final_cv_summary = (
    final_cv_metrics_df[["RMSE", "MAE", "R2", "Spearman"]]
    .agg(["mean", "std"]).T
)
display(final_cv_summary)

final_oof_metrics = metric_row(y_train, final_oof_pred)
print("Overall optimized-model OOF metrics:")
display(pd.Series(final_oof_metrics).to_frame("value"))

In [ ]:
print("=" * 60)
print("FINAL EVALUATION ON THE HELD-OUT TEST SET")
print("=" * 60)
print(f"RMSE: {test_metrics['RMSE']:.4f}")
print(f"MAE:  {test_metrics['MAE']:.4f}")
print(f"R2:   {test_metrics['R2']:.4f}")
print(f"Spearman: {test_metrics['Spearman']:.4f}")
print("=" * 60)

results_test = pd.DataFrame({
    "smiles": df_valid.loc[y_test.index, "smiles"].values,
    "docking_score_actual": y_test.values,
    "docking_score_predicted": y_pred,
})
results_test["residual"] = (
    results_test["docking_score_actual"] - results_test["docking_score_predicted"]
)
results_test["absolute_error"] = results_test["residual"].abs()
display(results_test.sort_values("absolute_error", ascending=False).head(10))

### How to read the main metrics

| metric | interpretation |
|---|---|
| **RMSE** | error in docking-score units; lower is better and large errors count more |
| **MAE** | average absolute error in docking-score units; lower is better |
| **R²** | fraction of held-out variation captured by the model; interpret with the dummy baseline and CI |
| **Spearman** | quality of molecular ranking; closer to +1 means better ordering |

Do not use a universal cutoff to label a docking-score model as good or bad. Interpret all validation controls together.

In [ ]:
print("QUICK INTERPRETATION")
print(f"Held-out RMSE: {test_metrics['RMSE']:.3f}")
print(f"Held-out MAE: {test_metrics['MAE']:.3f}")
print(f"Held-out R²: {test_metrics['R2']:.3f}")
print(f"Held-out Spearman: {test_metrics['Spearman']:.3f}")
print("Interpret with the dummy baseline, structural CV, bootstrap CI, Y-scrambling and AD.")

### Held-out metric uncertainty

The following bootstrap estimates 95% confidence intervals. For structural split modes, complete structural groups are resampled rather than individual molecules.

In [ ]:
def bootstrap_metric_intervals(
    y_true,
    y_pred,
    groups=None,
    n_bootstrap=2000,
    confidence=0.95,
    random_state=123,
):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rng = np.random.default_rng(random_state)

    samples = {k: [] for k in ["RMSE", "MAE", "R2", "Spearman"]}

    if groups is None:
        units = np.arange(len(y_true))
        group_positions = None
    else:
        groups = np.asarray(groups)
        units = np.unique(groups)
        group_positions = {g: np.flatnonzero(groups == g) for g in units}
        if len(units) < 10:
            warnings.warn(
                f"Only {len(units)} structural groups are present in the test set; "
                "cluster-bootstrap confidence intervals may be unstable.",
                RuntimeWarning,
            )

    for _bootstrap_i in range(n_bootstrap):
        if n_bootstrap >= 10 and (_bootstrap_i + 1) % max(1, n_bootstrap // 10) == 0:
            print(f"Bootstrap progress: {_bootstrap_i + 1}/{n_bootstrap}")
        sampled = rng.choice(units, size=len(units), replace=True)
        if group_positions is None:
            idx = sampled.astype(int)
        else:
            idx = np.concatenate([group_positions[g] for g in sampled])

        yt, yp = y_true[idx], y_pred[idx]
        if len(np.unique(yt)) < 2:
            continue

        vals = metric_row(yt, yp)
        for key, value in vals.items():
            if np.isfinite(value):
                samples[key].append(float(value))

    alpha = (1 - confidence) / 2
    rows = []
    for key, vals in samples.items():
        arr = np.asarray(vals)
        if len(arr) == 0:
            raise RuntimeError(
                f"No valid bootstrap samples were generated for metric {key}."
            )
        rows.append({
            "metric": key,
            "estimate": test_metrics[key],
            "ci_lower": np.quantile(arr, alpha),
            "ci_upper": np.quantile(arr, 1-alpha),
            "bootstrap_samples": len(arr),
        })
    return pd.DataFrame(rows)

test_bootstrap_groups = (
    None if split_groups_series is None
    else split_groups_series.loc[X_test.index].to_numpy()
)

test_metric_ci_df = bootstrap_metric_intervals(
    y_test,
    y_pred,
    groups=test_bootstrap_groups,
    n_bootstrap=BOOTSTRAP_REPEATS,
    confidence=BOOTSTRAP_CONFIDENCE,
    random_state=RANDOM_STATE,
)
display(test_metric_ci_df)

## Y-scrambling negative control

The molecular features are left unchanged while training docking scores are randomly permuted. The final model architecture and hyperparameters remain frozen. Strong performance on scrambled labels would indicate chance correlation, leakage or a validation artifact.

In [ ]:
def evaluate_y_scrambling(
    frozen_pipeline,
    X_data,
    y_data,
    cv,
    groups=None,
    repeats=30,
    random_state=123,
):
    rng = np.random.default_rng(random_state)
    Xr = X_data.reset_index(drop=True)
    yr = y_data.reset_index(drop=True)
    gv = None if groups is None else np.asarray(groups)
    rows = []

    for repeat in range(1, repeats + 1):
        if repeats >= 5 and (repeat == 1 or repeat % max(1, repeats // 5) == 0 or repeat == repeats):
            print(f"Y-scrambling progress: {repeat}/{repeats}")
        y_perm = pd.Series(rng.permutation(yr.to_numpy()))
        oof = np.empty(len(yr), dtype=float)

        for fit_idx, val_idx in cv.split(Xr, y_perm, gv):
            m = clone(frozen_pipeline)
            m.fit(Xr.iloc[fit_idx], y_perm.iloc[fit_idx])
            oof[val_idx] = m.predict(Xr.iloc[val_idx])

        rows.append({"repeat": repeat, **metric_row(y_perm, oof)})

    return pd.DataFrame(rows)

y_scrambling_df = evaluate_y_scrambling(
    final_model,
    X_train,
    y_train,
    cv_strategy,
    groups=cv_groups_train,
    repeats=Y_SCRAMBLING_REPEATS,
    random_state=RANDOM_STATE,
)

display(y_scrambling_df.describe().T)

plt.figure(figsize=(7, 4.5))
plt.hist(y_scrambling_df["R2"], bins=min(15, len(y_scrambling_df)), alpha=0.75)
plt.axvline(final_oof_metrics["R2"], linestyle="--", linewidth=2)
plt.xlabel("OOF R²")
plt.ylabel("Frequency")
plt.title("Y-scrambling negative control")
plt.show()

yscramble_p_r2 = (
    1 + np.sum(y_scrambling_df["R2"] >= final_oof_metrics["R2"])
) / (1 + len(y_scrambling_df))

print(f"Real optimized-model OOF R²: {final_oof_metrics['R2']:.4f}")
print(f"Mean scrambled OOF R²: {y_scrambling_df['R2'].mean():.4f}")
print(f"Empirical one-sided p-value: {yscramble_p_r2:.4f}")
minimum_resolvable_p = 1 / (1 + Y_SCRAMBLING_REPEATS)
print(
    f"Minimum resolvable empirical p-value with "
    f"{Y_SCRAMBLING_REPEATS} permutations: {minimum_resolvable_p:.4f}"
)

In [ ]:
ranking_metrics_df = top_k_ranking_metrics(y_test, y_pred)

print("=" * 60)
print("TOP-K RANKING METRICS")
print("=" * 60)
display(ranking_metrics_df)

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.7)

min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], "r--")

plt.xlabel("Docking score real")
plt.ylabel("Docking score predito")
plt.title("Predição de Docking score no set teste")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(x=y_test, y=y_pred, scatter_kws={"alpha": 0.7})
plt.xlabel("Docking score real")
plt.ylabel("Docking score predito")
plt.title("Predição de Docking score no set teste")
plt.show()

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
sns.histplot(residuals, kde=True)
plt.xlabel("Residuo: Actual docking score - Predicted docking score")
plt.title("Distribuição dos resíduos")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted docking score")
plt.ylabel("Residual")
plt.title("Residuals vs predicted docking score")
plt.show()

## 9.2 Publication-ready evaluation figures

The cells below regenerate the main model-evaluation figures using a consistent visual style and automatically save them as `.png` and `.pdf` files in `FIGURES_DIR`.

In [ ]:
set_publication_style()

# Actual vs predicted docking score
fig, ax = plt.subplots(figsize=(7.2, 7.0))
ax.scatter(y_test, y_pred, alpha=0.55, s=28, color="#2C7FB8", edgecolor="white", linewidth=0.25)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
ax.plot([min_val, max_val], [min_val, max_val], color="#D62728", linestyle="--", linewidth=1.8, label="Ideal prediction")
ax.set_xlabel("Actual docking score (kcal/mol)")
ax.set_ylabel("Predicted docking score (kcal/mol)")
ax.set_title("Actual vs Predicted Docking Scores", pad=12, weight="bold")
ax.legend(frameon=True)
polish_axes(ax)
fig.tight_layout()
save_figure(fig, "actual_vs_predicted_docking_score")
plt.show()

# Residual distribution
residuals = y_test - y_pred
fig, ax = plt.subplots(figsize=(8.5, 5.5))
sns.histplot(residuals, kde=True, bins=40, color="#4C78A8", edgecolor="black", linewidth=0.4, ax=ax)
ax.axvline(0, color="#D62728", linestyle="--", linewidth=1.6, label="Zero residual")
ax.set_xlabel("Residual: actual docking score - predicted docking score")
ax.set_ylabel("Frequency")
ax.set_title("Residual Distribution", pad=12, weight="bold")
ax.legend(frameon=True)
polish_axes(ax)
fig.tight_layout()
save_figure(fig, "residual_distribution")
plt.show()

# Residuals vs predicted
fig, ax = plt.subplots(figsize=(8.5, 5.5))
ax.scatter(y_pred, residuals, alpha=0.55, s=28, color="#59A14F", edgecolor="white", linewidth=0.25)
ax.axhline(0, color="#D62728", linestyle="--", linewidth=1.6)
ax.set_xlabel("Predicted docking score (kcal/mol)")
ax.set_ylabel("Residual (kcal/mol)")
ax.set_title("Residuals vs Predicted Docking Scores", pad=12, weight="bold")
polish_axes(ax)
fig.tight_layout()
save_figure(fig, "residuals_vs_predicted")
plt.show()

## 10. Main feature analysis

Because the final model is a `Pipeline`, we first recover the feature names after preprocessing. Then we associate those names with the feature-importance values computed by the selected tree-based model.

In [ ]:
def get_feature_names_from_preprocessor(fitted_pipeline):
    preprocessor = fitted_pipeline.named_steps["preprocess"]

    continuous_features = CONTINUOUS_COLS

    selected_fp_mask = (
        preprocessor
        .named_transformers_["fingerprints"]
        .named_steps["variance"]
        .get_support()
    )
    selected_fp_features = np.array(fp_cols)[selected_fp_mask].tolist()

    return continuous_features + selected_fp_features


def get_selected_model_importances(fitted_pipeline, model_type):
    model = fitted_pipeline.named_steps["model"]

    if model_type == "lightgbm":
        return model.booster_.feature_importance(importance_type="gain")

    if model_type == "catboost":
        return model.get_feature_importance(type="FeatureImportance")

    if model_type == "xgboost":
        return model.feature_importances_

    if hasattr(model, "feature_importances_"):
        return model.feature_importances_

    raise ValueError(f"Feature importance is not implemented for model type: {model_type}")


feature_names = get_feature_names_from_preprocessor(final_model)
importances = get_selected_model_importances(final_model, FINAL_MODEL_TYPE)

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False)

print("=" * 60)
print("MOST IMPORTANT FEATURES")
print("=" * 60)
print(f"Features after VarianceThreshold: {len(feature_importance_df)}")
display(feature_importance_df.head(20))

In [ ]:
top_n = 10
top_features = feature_importance_df.head(top_n)

plt.figure(figsize=(10, 7))
sns.barplot(data=top_features, x="importance", y="feature", palette="viridis")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title(f"Top {top_n} most important features for docking score prediction")
plt.tight_layout()
plt.show()

In [ ]:
feature_importance_df["type"] = np.where(
    feature_importance_df["feature"].str.startswith(fingerprint_prefixes),
    "Molecular fingerprint",
    "Physicochemical descriptor"
)

importance_by_type = (
    feature_importance_df
    .groupby("type")["importance"]
    .sum()
    .sort_values(ascending=False)
)

display(importance_by_type.to_frame("total_importance"))

plt.figure(figsize=(7, 5))
importance_by_type.plot(kind="bar")
plt.ylabel("Total importance")
plt.title("Aggregated importance by feature type")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Publication-ready feature importance plot
set_publication_style()
top_n = 10
top_features = feature_importance_df.head(top_n).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7.5))
sns.barplot(
    data=top_features,
    x="importance",
    y="feature",
    palette="viridis",
    edgecolor="black",
    linewidth=0.4,
    ax=ax,
)
ax.set_xlabel("Model feature importance")
ax.set_ylabel("Feature")
ax.set_title(f"Top {top_n} Features for docking score prediction", pad=12, weight="bold")
polish_axes(ax)
fig.tight_layout()
save_figure(fig, f"top_{top_n}_model_feature_importance_{FINAL_MODEL_TYPE}")
plt.show()

# 10.0.1 Correlation heatmap after VarianceThreshold

In [ ]:
TOP_N_CORR_FEATURES = 25  # adjust if needed

print("=" * 60)
print("CORRELATION HEATMAP AFTER VARIANCETHRESHOLD")
print("=" * 60)

feature_names_after_preprocessing = get_feature_names_from_preprocessor(final_model)

X_test_transformed = final_model.named_steps["preprocess"].transform(X_test)

if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

X_test_transformed_df = pd.DataFrame(
    X_test_transformed,
    columns=feature_names_after_preprocessing
)

top_features_for_corr = (
    feature_importance_df
    .head(TOP_N_CORR_FEATURES)["feature"]
    .tolist()
)

top_features_for_corr = [
    feature for feature in top_features_for_corr
    if feature in X_test_transformed_df.columns
]

corr_matrix_top_features = X_test_transformed_df[top_features_for_corr].corr()

plt.figure(figsize=(12, 10), dpi=300)

sns.heatmap(
    corr_matrix_top_features,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.35,
    linecolor="white",
    cbar_kws={"label": "Pearson correlation"}
)

plt.title(
    "Correlation heatmap of top model features after VarianceThreshold",
    fontsize=14,
    fontweight="bold"
)

plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()

heatmap_path = FIGURES_DIR / "top_features_correlation_after_variancethreshold.png"
plt.savefig(heatmap_path, dpi=600, bbox_inches="tight")
plt.show()

announce_saved(heatmap_path, "correlation heatmap after VarianceThreshold")

Morgan features such as `MORGAN_123` are hashed structural-environment bits. Because hashing collisions are possible, one bit can correspond to more than one observed atom environment.

The notebook maps the most important Morgan bits back to representative RDKit environments found in the **training set**, keeping multiple unique examples when needed.

### 10.1.1 Permutation importance

Permutation importance measures how much performance worsens when a feature is shuffled. If shuffling a feature greatly increases the error, that feature was important for the model.

In [ ]:
sample_size = min(2000, len(X_test))

X_test_perm = X_test.sample(sample_size, random_state=RANDOM_STATE)
y_test_perm = y_test.loc[X_test_perm.index]

perm_result = permutation_importance(
    final_model,
    X_test_perm,
    y_test_perm,
    scoring="neg_root_mean_squared_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=MODEL_N_JOBS
)

perm_importance_df = pd.DataFrame({
    "feature": X_test_perm.columns,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std,
}).sort_values("importance_mean", ascending=False)

print("=" * 60)
print("PERMUTATION IMPORTANCE")
print("=" * 60)
display(perm_importance_df.head(10))

In [ ]:
top_perm = perm_importance_df.head(10)

plt.figure(figsize=(10, 7))
sns.barplot(data=top_perm, x="importance_mean", y="feature", palette="mako")
plt.xlabel("Mean permutation importance")
plt.ylabel("Feature")
plt.title("Top 10 features by permutation importance")
plt.tight_layout()
plt.show()

In [ ]:
# Publication-ready permutation importance plot
set_publication_style()
top_n = 10
top_perm_pub = perm_importance_df.head(top_n).sort_values("importance_mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, 7.5))
sns.barplot(
    data=top_perm_pub,
    x="importance_mean",
    y="feature",
    palette="mako",
    edgecolor="black",
    linewidth=0.4,
    ax=ax,
)
ax.set_xlabel("Mean permutation importance")
ax.set_ylabel("Feature")
ax.set_title(f"Top {top_n} Features by Permutation Importance", pad=12, weight="bold")
polish_axes(ax)
fig.tight_layout()
save_figure(fig, f"top_{top_n}_permutation_importance_{FINAL_MODEL_TYPE}")
plt.show()

### 10.1.2 Map important Morgan bits to representative chemical substructures

The mapping scans only training molecules. For each important `MORGAN_<bit>`, RDKit `bitInfo` is used to recover atom environments that generated the bit.

Because Morgan fingerprints are hashed, collisions are possible. Multiple distinct representative substructures are therefore retained for each bit.

In [ ]:
def morgan_environment_smiles(mol, atom_idx, radius):
    if radius == 0:
        return Chem.MolFragmentToSmiles(
            mol, atomsToUse=[int(atom_idx)], canonical=True
        )

    bond_ids = list(
        Chem.FindAtomEnvironmentOfRadiusN(mol, int(radius), int(atom_idx))
    )
    if not bond_ids:
        return Chem.MolFragmentToSmiles(
            mol, atomsToUse=[int(atom_idx)], canonical=True
        )

    submol = Chem.PathToSubmol(mol, bond_ids)
    return Chem.MolToSmiles(submol, canonical=True)

def map_morgan_bits_to_substructures(
    feature_importance_df,
    training_smiles,
    fp_config,
    top_n_bits=10,
    max_unique_examples=5,
):
    morgan_rows = feature_importance_df[
        feature_importance_df["feature"].str.startswith("MORGAN_")
    ].head(top_n_bits)

    radius = int(fp_config["morgan_radius"])
    n_bits = int(fp_config["morgan_bits"])
    records = []

    for _, feature_row in morgan_rows.iterrows():
        feature_name = feature_row["feature"]
        bit_id = int(feature_name.split("_")[1])
        examples, source_smiles = [], []

        for smiles in training_smiles:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is None:
                continue

            bit_info = {}
            AllChem.GetMorganFingerprintAsBitVect(
                mol, radius, nBits=n_bits, bitInfo=bit_info
            )

            for atom_idx, env_radius in bit_info.get(bit_id, []):
                env_smiles = morgan_environment_smiles(
                    mol, atom_idx, env_radius
                )
                if env_smiles not in examples:
                    examples.append(env_smiles)
                    source_smiles.append(smiles)

                if len(examples) >= max_unique_examples:
                    break

            if len(examples) >= max_unique_examples:
                break

        records.append({
            "feature": feature_name,
            "bit_id": bit_id,
            "importance_mean": feature_row.get("importance_mean", np.nan),
            "n_unique_representative_environments": len(examples),
            "representative_substructures": " | ".join(examples),
            "example_training_smiles": " | ".join(source_smiles),
            "hash_collision_possible": True,
        })

    return pd.DataFrame(records)

training_smiles_for_mapping = df_valid.loc[X_train.index, "smiles"].tolist()

morgan_bit_mapping_df = map_morgan_bits_to_substructures(
    perm_importance_df,
    training_smiles_for_mapping,
    FINGERPRINT_CONFIG,
    top_n_bits=10,
    max_unique_examples=5,
)

display(morgan_bit_mapping_df)
morgan_bit_mapping_df.to_csv(MORGAN_MAPPING_CSV, index=False)
announce_saved(MORGAN_MAPPING_CSV, "Morgan-bit to substructure mapping (CSV)")

### 10.1.3 SHAP global and local interpretation

SHAP contribution values are computed for the selected final tree model. Native contribution methods are used for XGBoost, LightGBM and CatBoost; `shap.TreeExplainer` is used for RandomForest and ExtraTrees.

The reconstruction check verifies that feature contributions plus the base value reproduce the model prediction.

**Important:** SHAP and permutation importance are post-hoc interpretation only. Results from the held-out test set must not be used to return to feature/model selection, otherwise the test set would cease to be a final independent evaluation.

> **Interpretability hygiene:** explanations computed on the held-out test set are post-hoc diagnostics only. Do not use test-set SHAP/permutation results to change feature selection, model family or hyperparameters and then re-report performance on the same test set. Any such iteration would turn the test set into part of model development.

In [ ]:
import shap

sample_size = min(1000, len(X_test))
X_explain = X_test.sample(sample_size, random_state=RANDOM_STATE)

preprocessor = final_model.named_steps["preprocess"]
selected_model = final_model.named_steps["model"]

X_explain_transformed = preprocessor.transform(X_explain)

if hasattr(X_explain_transformed, "toarray"):
    X_explain_transformed = X_explain_transformed.toarray()

feature_names = get_feature_names_from_preprocessor(final_model)
X_explain_transformed_df = pd.DataFrame(
    X_explain_transformed,
    columns=feature_names
)

if FINAL_MODEL_TYPE == "xgboost":
    import xgboost as xgb
    booster = selected_model.get_booster()
    xgb_matrix = xgb.DMatrix(
        X_explain_transformed_df,
        feature_names=feature_names
    )
    shap_contribs = booster.predict(xgb_matrix, pred_contribs=True)
    shap_values = shap_contribs[:, :-1]
    shap_base_values = shap_contribs[:, -1]

elif FINAL_MODEL_TYPE == "lightgbm":
    shap_contribs = selected_model.predict(X_explain_transformed_df, pred_contrib=True)
    shap_values = shap_contribs[:, :-1]
    shap_base_values = shap_contribs[:, -1]

elif FINAL_MODEL_TYPE == "catboost":
    from catboost import Pool
    cat_pool = Pool(X_explain_transformed_df, feature_names=feature_names)
    shap_contribs = selected_model.get_feature_importance(
        data=cat_pool,
        type="ShapValues"
    )
    shap_values = shap_contribs[:, :-1]
    shap_base_values = shap_contribs[:, -1]

elif FINAL_MODEL_TYPE in {"random_forest", "extra_trees"}:
    explainer = shap.TreeExplainer(selected_model)
    shap_explanation = explainer(X_explain_transformed_df)
    shap_values = np.asarray(shap_explanation.values)
    base_values = np.asarray(shap_explanation.base_values)
    flat_base_values = base_values.reshape(-1)
    if flat_base_values.size == 1:
        shap_base_values = np.full(
            len(X_explain_transformed_df),
            float(flat_base_values[0]),
        )
    elif flat_base_values.size == len(X_explain_transformed_df):
        shap_base_values = flat_base_values
    else:
        raise RuntimeError(
            "Unexpected SHAP base-value shape for tree regressor: "
            f"{base_values.shape}"
        )

else:
    raise ValueError(f"SHAP contributions are not implemented for: {FINAL_MODEL_TYPE}")

# Sanity check: feature contributions + bias should reconstruct the model prediction.
reconstructed_pred = shap_values.sum(axis=1) + shap_base_values
pipeline_pred = final_model.predict(X_explain)
max_reconstruction_difference = np.max(
    np.abs(reconstructed_pred - pipeline_pred)
)
print("Selected model:", FINAL_MODEL_TYPE)
print("Max reconstruction difference:", max_reconstruction_difference)

if not np.allclose(reconstructed_pred, pipeline_pred, rtol=1e-5, atol=1e-5):
    raise RuntimeError(
        "SHAP contributions failed to reconstruct model predictions within tolerance."
    )

In [ ]:
shap.summary_plot(
    shap_values,
    X_explain_transformed_df,
    plot_type="bar",
    max_display=10
)

In [ ]:
shap.summary_plot(
    shap_values,
    X_explain_transformed_df,
    max_display=15
)

The first plot shows the mean global feature importance. The second combines importance and effect direction: it shows whether high or low values of a feature tend to increase or decrease the predicted `docking score`.

### 10.1.4 SHAP local explanation for one prediction

This cell helps explain one specific fragment from the test set.

In [ ]:
local_idx = min(80, len(X_explain) - 1)

original_idx = X_explain.index[local_idx]
smiles_local = df_valid.loc[original_idx, "smiles"]
real_docking_score = y.loc[original_idx]
pred_docking_score = final_model.predict(X_explain.iloc[[local_idx]])[0]

print("=" * 60)
print("LOCAL EXPLANATION")
print("=" * 60)
print(f"Original index: {original_idx}")
print(f"SMILES: {smiles_local}")
print(f"Actual docking score: {real_docking_score:.4f}")
print(f"Predicted docking score: {pred_docking_score:.4f}")
print("=" * 60)

shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[local_idx],
        base_values=shap_base_values[local_idx],
        data=X_explain_transformed_df.iloc[local_idx].values,
        feature_names=feature_names,
    ),
    max_display=20,
)

## 11. Applicability domain by Morgan/Tanimoto similarity

For each query molecule, the notebook computes its maximum Morgan/Tanimoto similarity to a deterministic reference subset of the training molecules.

The threshold is data-derived: it is the selected lower quantile of leave-one-out nearest-neighbor similarities within the training reference. Predictions below that threshold are flagged as outside the applicability domain.

To keep the quadratic reference calibration bounded on very large datasets, the reference set can be capped using a seeded random sample. The exact reference SMILES and threshold are serialized with the model.

### In plain language: applicability domain

The model is generally easier to interpret when a query molecule resembles chemistry represented during training. Morgan/Tanimoto similarity measures this structural proximity.

- higher similarity: closer to known training chemistry;
- lower similarity: stronger extrapolation;
- outside AD: keep the prediction, but interpret it with additional caution.

The AD flag is not a probability of activity and not a guarantee of correctness.

In [ ]:
#@title Applicability-domain settings
AD_REFERENCE_MAX = 3000 #@param {type:"integer"}
AD_LOWER_QUANTILE = 0.05 #@param {type:"number"}

def smiles_to_morgan_bitvect(smiles, fp_config):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return AllChem.GetMorganFingerprintAsBitVect(
        mol,
        int(fp_config["morgan_radius"]),
        nBits=int(fp_config["morgan_bits"]),
    )

def deterministic_reference_smiles(smiles_series, max_size, random_state):
    s = pd.Series(smiles_series).dropna().astype(str).reset_index(drop=True)
    if len(s) <= max_size:
        return s.tolist()
    rng = np.random.default_rng(random_state)
    selected = np.sort(rng.choice(len(s), size=max_size, replace=False))
    return s.iloc[selected].tolist()

def build_fp_reference(smiles_list, fp_config):
    valid_smiles, fps = [], []
    for smiles in smiles_list:
        fp = smiles_to_morgan_bitvect(smiles, fp_config)
        if fp is not None:
            valid_smiles.append(smiles)
            fps.append(fp)
    return valid_smiles, fps

def leave_one_out_max_tanimoto(reference_fps):
    if len(reference_fps) < 2:
        raise ValueError("At least two valid reference molecules are required.")
    nearest = np.empty(len(reference_fps), dtype=float)
    for i, fp in enumerate(reference_fps):
        sims = np.asarray(
            DataStructs.BulkTanimotoSimilarity(fp, reference_fps),
            dtype=float,
        )
        sims[i] = -np.inf
        nearest[i] = np.max(sims)
    return nearest

def max_tanimoto_to_reference(smiles_list, reference_fps, fp_config):
    out = []
    for smiles in smiles_list:
        fp = smiles_to_morgan_bitvect(smiles, fp_config)
        if fp is None:
            out.append(np.nan)
            continue
        sims = DataStructs.BulkTanimotoSimilarity(fp, reference_fps)
        out.append(float(max(sims)) if sims else np.nan)
    return np.asarray(out, dtype=float)

train_smiles_all = df_valid.loc[X_train.index, "smiles"].tolist()
ad_reference_smiles = deterministic_reference_smiles(
    train_smiles_all, AD_REFERENCE_MAX, RANDOM_STATE
)
ad_reference_smiles, ad_reference_fps = build_fp_reference(
    ad_reference_smiles, FINGERPRINT_CONFIG
)

ad_reference_nn_similarity = leave_one_out_max_tanimoto(ad_reference_fps)
ad_threshold = float(
    np.quantile(ad_reference_nn_similarity, AD_LOWER_QUANTILE)
)

test_smiles = df_valid.loc[X_test.index, "smiles"].tolist()
test_max_tanimoto = max_tanimoto_to_reference(
    test_smiles, ad_reference_fps, FINGERPRINT_CONFIG
)

results_test["max_tanimoto_to_ad_reference"] = test_max_tanimoto
results_test["within_applicability_domain"] = (
    results_test["max_tanimoto_to_ad_reference"] >= ad_threshold
)

print("=" * 60)
print("APPLICABILITY DOMAIN")
print("=" * 60)
print(f"Training molecules: {len(train_smiles_all)}")
print(f"Reference molecules used: {len(ad_reference_smiles)}")
print(f"Lower quantile: {AD_LOWER_QUANTILE:.3f}")
print(f"Data-derived Tanimoto threshold: {ad_threshold:.4f}")
print(f"Test molecules inside AD: {results_test['within_applicability_domain'].mean():.1%}")
print("=" * 60)

display(
    results_test.sort_values(
        ["within_applicability_domain", "max_tanimoto_to_ad_reference"],
        ascending=[True, True],
    ).head(20)
)

if AD_REFERENCE_MAX > 5000:
    warnings.warn(
        "AD_REFERENCE_MAX > 5000 may become expensive because the leave-one-out "
        "nearest-neighbor calibration scales approximately quadratically."
    )

if AD_REFERENCE_MAX < 100:
    warnings.warn(
        "A very small AD reference set may give an unstable estimate of the "
        "training chemical-space similarity distribution."
    )


### Empirical validation of the applicability domain

The continuous Tanimoto similarity is analyzed against held-out prediction error. This tests whether predictions actually become less reliable as molecules move away from the training reference space.

In [ ]:
ad_test_analysis = results_test[
    [
        "docking_score_actual",
        "docking_score_predicted",
        "absolute_error",
        "max_tanimoto_to_ad_reference",
        "within_applicability_domain",
    ]
].dropna().copy()

if len(ad_test_analysis) >= 5:
    similarity_error_spearman = spearmanr(
        ad_test_analysis["max_tanimoto_to_ad_reference"],
        ad_test_analysis["absolute_error"],
    )

    print(
        "Spearman(max Tanimoto, absolute error): "
        f"{similarity_error_spearman.correlation:.4f}; "
        f"p={similarity_error_spearman.pvalue:.4g}"
    )

    plt.figure(figsize=(7, 5))
    plt.scatter(
        ad_test_analysis["max_tanimoto_to_ad_reference"],
        ad_test_analysis["absolute_error"],
        alpha=0.7,
    )
    plt.axvline(ad_threshold, linestyle="--", linewidth=1.5)
    plt.xlabel("Maximum Tanimoto similarity to AD reference")
    plt.ylabel("Absolute docking-score prediction error")
    plt.title("Prediction error vs chemical-space proximity")
    plt.show()

    similarity_bins = pd.cut(
        ad_test_analysis["max_tanimoto_to_ad_reference"],
        bins=[0.0, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.000001],
        include_lowest=True,
    )

    rows = []
    for bin_name, g in ad_test_analysis.assign(similarity_bin=similarity_bins).groupby(
        "similarity_bin", observed=False
    ):
        if len(g) == 0:
            continue
        rows.append({
            "similarity_bin": str(bin_name),
            "n": len(g),
            "mean_similarity": g["max_tanimoto_to_ad_reference"].mean(),
            "MAE": g["absolute_error"].mean(),
            "RMSE": np.sqrt(np.mean(
                (g["docking_score_actual"] - g["docking_score_predicted"]) ** 2
            )),
            "inside_AD_fraction": g["within_applicability_domain"].mean(),
        })

    ad_binned_performance = pd.DataFrame(rows)
    display(ad_binned_performance)
else:
    print("Too few held-out observations for stable AD diagnostics.")

In [ ]:
ad_error_summary = (
    results_test
    .groupby("within_applicability_domain", dropna=False)
    .agg(
        n=("absolute_error", "size"),
        mean_absolute_error=("absolute_error", "mean"),
        median_absolute_error=("absolute_error", "median"),
        mean_max_tanimoto=("max_tanimoto_to_ad_reference", "mean"),
    )
)
display(ad_error_summary)

## 12. Reproducibility metadata and model serialization

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def package_version(package_name):
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return None

environment_versions = {
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": package_version("scikit-learn"),
    "scipy": package_version("scipy"),
    "rdkit": rdBase.rdkitVersion,
    "xgboost": package_version("xgboost"),
    "lightgbm": package_version("lightgbm"),
    "catboost": package_version("catboost"),
    "optuna": package_version("optuna"),
    "shap": package_version("shap"),
}

run_config = {
    "molflood_version": MOLFLOOD_VERSION,
    "molflood_release_status": MOLFLOOD_RELEASE_STATUS,
    "molflood_authors": MOLFLOOD_AUTHORS,
    "execution_mode": "publication_only",
    "random_state": RANDOM_STATE,
    "reproducible_mode": REPRODUCIBLE_MODE,
    "n_jobs": N_JOBS,
    "split_mode": SPLIT_MODE,
    "test_size": TEST_SIZE,
    "cv_folds": CV_FOLDS,
    "cv_strategy": type(cv_strategy).__name__,
    "fingerprint_preset": FINGERPRINT_PRESET,
    "fingerprint_config": FINGERPRINT_CONFIG,
    "selected_algorithm": FINAL_MODEL_TYPE,
    "optuna_trials": OPTUNA_TRIALS,
    "variance_threshold_max": VARIANCE_THRESHOLD_MAX,
    "ad_reference_max": AD_REFERENCE_MAX,
    "ad_lower_quantile": AD_LOWER_QUANTILE,
    "ad_threshold": ad_threshold,
    "y_scrambling_repeats": Y_SCRAMBLING_REPEATS,
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "bootstrap_confidence": BOOTSTRAP_CONFIDENCE,
    "publication_mode": PUBLICATION_MODE,
}

reproducibility_record = {
    "dataset_sha256": sha256_file(DATA_CSV),
    "target_metadata": TARGET_METADATA,
    "environment": environment_versions,
    "run_config": run_config,
}

with open(ENVIRONMENT_JSON, "w", encoding="utf-8") as handle:
    json.dump(reproducibility_record, handle, indent=2, default=str)

announce_saved(ENVIRONMENT_JSON, "environment/reproducibility metadata (JSON)")
display(reproducibility_record)

In [ ]:
REQUIREMENTS_LOCK = BASE_DIR / "requirements-lock.txt"
LOCK_PACKAGES = [
    "numpy", "pandas", "scipy", "matplotlib", "seaborn",
    "scikit-learn", "rdkit", "xgboost", "lightgbm",
    "catboost", "optuna", "joblib", "shap", "tabulate",
]

lock_lines = []
for package_name in LOCK_PACKAGES:
    version = package_version(package_name)
    if version is None:
        if PUBLICATION_MODE:
            raise RuntimeError(
                f"Cannot generate publication lock: '{package_name}' is not installed."
            )
        continue
    lock_lines.append(f"{package_name}=={version}")

REQUIREMENTS_LOCK.write_text("\n".join(lock_lines) + "\n", encoding="utf-8")
announce_saved(REQUIREMENTS_LOCK, "exact Python dependency lock")
print(REQUIREMENTS_LOCK.read_text(encoding="utf-8"))

In [ ]:
TARGET_METADATA_JSON = BASE_DIR / "target_metadata.json"

with open(TARGET_METADATA_JSON, "w", encoding="utf-8") as handle:
    json.dump(TARGET_METADATA, handle, indent=2, ensure_ascii=False, default=str)

announce_saved(TARGET_METADATA_JSON, "biological target/disease metadata (JSON)")

The serialized artifact contains the fitted model, selection metadata, applicability-domain reference and reproducibility record.

In [ ]:
artifact = {
    "molflood_version": MOLFLOOD_VERSION,
    "molflood_release_status": MOLFLOOD_RELEASE_STATUS,
    "molflood_authors": MOLFLOOD_AUTHORS,
    "model": final_model,
    "target_metadata": TARGET_METADATA,
    "continuous_cols": CONTINUOUS_COLS,
    "fp_cols": fp_cols,
    "fingerprint_config": FINGERPRINT_CONFIG,
    "fingerprint_preset": FINGERPRINT_PRESET,
    "split_mode": SPLIT_MODE,
    "cv_strategy": type(cv_strategy).__name__,
    "random_state": RANDOM_STATE,
    "test_metrics": test_metrics,
    "best_params": study.best_params,
    "selected_algorithm": FINAL_MODEL_TYPE,
    "algorithm_selection_cv_results": baseline_df.to_dict(),
    "cv_fold_diagnostics": cv_fold_diagnostics.to_dict(orient="records"),
    "final_cv_metrics_by_fold": final_cv_metrics_df.to_dict(orient="records"),
    "final_oof_metrics": final_oof_metrics,
    "test_metric_confidence_intervals": test_metric_ci_df.to_dict(orient="records"),
    "y_scrambling_summary": y_scrambling_df.describe().to_dict(),
    "y_scrambling_empirical_p_r2": yscramble_p_r2,
    "applicability_domain": {
        "metric": "Morgan fingerprint Tanimoto similarity",
        "threshold": ad_threshold,
        "lower_quantile": AD_LOWER_QUANTILE,
        "reference_smiles": ad_reference_smiles,
        "reference_size": len(ad_reference_smiles),
    },
    "reproducibility": reproducibility_record,
}

joblib.dump(artifact, MODEL_PATH)
announce_saved(MODEL_PATH, "trained docking-score model (.joblib)")

MODEL_SHA256 = sha256_file(MODEL_PATH)
MODEL_SHA256_PATH = BASE_DIR / "docking_score_model.sha256"
MODEL_SHA256_PATH.write_text(
    f"{MODEL_SHA256}  {MODEL_PATH.name}\n",
    encoding="utf-8",
)
announce_saved(MODEL_SHA256_PATH, "model SHA-256 checksum")
print("Model SHA-256:", MODEL_SHA256)

### Automatically generated model card

For public use, every trained model should travel with its biological context, docking provenance, validation strategy, performance and applicability-domain definition. The following cell generates a Markdown model card from the actual run metadata.

In [ ]:
MODEL_CARD_PATH = BASE_DIR / "MODEL_CARD.md"


def format_metadata_value(value):
    if value is None:
        return "Not reported"
    if isinstance(value, (list, tuple)):
        return "; ".join(map(str, value))
    if isinstance(value, dict):
        return json.dumps(value, ensure_ascii=False)
    return str(value)


model_card = f"""# Model Card — {TARGET_METADATA['campaign_title']}

**MolFlood version:** {MOLFLOOD_VERSION} ({MOLFLOOD_RELEASE_STATUS})
**Developed by:** {", ".join(MOLFLOOD_AUTHORS)}

## Scope

**Predictive endpoint:** docking score
**Disease:** {format_metadata_value(TARGET_METADATA['disease_name'])}
**Biological target:** {format_metadata_value(TARGET_METADATA['target_name'])}
**Organism:** {format_metadata_value(TARGET_METADATA['target_organism'])}
**PDB ID:** {format_metadata_value(TARGET_METADATA['pdb_id'])}

This model predicts docking scores generated in the computational context described below. It does not directly predict experimental binding affinity or therapeutic efficacy.

## Intended use

{format_metadata_value(TARGET_METADATA['intended_use'])}

## Docking provenance

- Software: {format_metadata_value(TARGET_METADATA['docking_software'])}
- Software version: {format_metadata_value(TARGET_METADATA['docking_software_version'])}
- Protocol ID: {format_metadata_value(TARGET_METADATA['docking_protocol_id'])}
- Score: docking_score
- Units: {format_metadata_value(TARGET_METADATA['docking_score_units'])}
- Direction: {format_metadata_value(TARGET_METADATA['docking_score_direction'])}
- Binding-site definition: {format_metadata_value(TARGET_METADATA['binding_site_definition'])}
- Grid center: {format_metadata_value(TARGET_METADATA['grid_center'])}
- Grid size: {format_metadata_value(TARGET_METADATA['grid_size'])}
- Exhaustiveness: {format_metadata_value(TARGET_METADATA['exhaustiveness'])}

## Molecular representation

- Fingerprint preset: {FINGERPRINT_PRESET}
- Morgan radius: {FINGERPRINT_CONFIG['morgan_radius']}
- Morgan bits: {FINGERPRINT_CONFIG['morgan_bits']}
- RDKit continuous descriptors: {len(CONTINUOUS_COLS)}

## Dataset and validation

- Total valid molecules: {len(X)}
- Training molecules: {len(X_train)}
- Held-out test molecules: {len(X_test)}
- Split: {SPLIT_MODE}
- Held-out test fraction: {TEST_SIZE}
- Cross-validation: {type(cv_strategy).__name__}
- CV folds: {CV_FOLDS}
- Selected algorithm: {FINAL_MODEL_TYPE}
- DummyMean CV RMSE: {baseline_df.loc['DummyMean', 'RMSE']:.6f}
- Selected-model CV RMSE before Optuna: {baseline_df.loc[best_baseline_name, 'RMSE']:.6f}
- Relative RMSE improvement over DummyMean: {relative_rmse_improvement_vs_dummy:.2%}
- Optuna trials: {OPTUNA_TRIALS}

## Held-out test performance

- RMSE: {test_metrics['RMSE']:.6f}
- MAE: {test_metrics['MAE']:.6f}
- R²: {test_metrics['R2']:.6f}
- Spearman: {test_metrics['Spearman']:.6f}

### 95% bootstrap confidence intervals

{test_metric_ci_df.to_markdown(index=False)}

### Optimized-model cross-validation by fold

{final_cv_metrics_df.to_markdown(index=False)}

### Y-scrambling negative control

- Repetitions: {Y_SCRAMBLING_REPEATS}
- Real optimized-model OOF R²: {final_oof_metrics['R2']:.6f}
- Mean scrambled OOF R²: {y_scrambling_df['R2'].mean():.6f}
- Empirical one-sided p-value: {yscramble_p_r2:.6f}

## Applicability domain

- Similarity: Morgan/Tanimoto
- Reference molecules: {len(ad_reference_smiles)}
- Threshold: {ad_threshold:.6f}
- Threshold definition: lower {AD_LOWER_QUANTILE:.1%} quantile of leave-one-out nearest-neighbor similarities in the training reference set

- Held-out coverage inside AD: {results_test['within_applicability_domain'].mean():.2%}

Predictions outside this domain should be interpreted with increased caution.

## Reproducibility

- Random seed: {RANDOM_STATE}
- Reproducible mode: {REPRODUCIBLE_MODE}
- Publication mode: {PUBLICATION_MODE}
- Exact dependencies: `requirements-lock.txt`
- Dataset SHA-256: {reproducibility_record['dataset_sha256']}
- Model SHA-256: {MODEL_SHA256}
- Environment metadata: `environment_versions.json`

## Out of scope

{chr(10).join('- ' + item for item in TARGET_METADATA['out_of_scope'])}

## Scientific interpretation

The model approximates the docking-score function represented by its training data and docking protocol. A favorable predicted docking score is a prioritization signal for computational screening, not evidence of biological activity, clinical efficacy, or experimental binding.
"""

MODEL_CARD_PATH.write_text(model_card, encoding="utf-8")
announce_saved(MODEL_CARD_PATH, "public model card (Markdown)")
print(model_card)

## 13. Path B — Predict docking score for external new molecules

This step is for a `.csv` file containing new SMILES that **were not used in training or testing**. It should be executed after the `.joblib` model has been saved.

The external CSV must contain at least a `smiles` column. Other columns, such as name, ID, library, supplier or molecule source, are preserved in the output. The output preserves the same order as the input file for straightforward downstream prioritization and analysis.

Minimal example:

```text
id,smiles
frag_001,c1ccccc1
frag_002,CCOc1ccccc1
```

**Security note:** the `.joblib` format relies on Python pickle semantics. Load only model artifacts obtained from a trusted source and verify the published SHA-256 checksum before use.

### If you only want predictions

Use a trusted model produced by this workflow, verify its SHA-256 checksum, provide a CSV with `smiles`, and run this section. Output includes docking-score predictions and applicability-domain information.

In [ ]:
# Inference-only helpers for external docking-score prediction.
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem, DataStructs

DEFAULT_CONTINUOUS_COLS_FOR_INFERENCE = [
    "MolWt", "ExactMolWt", "MolLogP", "TPSA", "MolMR",
    "NumHDonors", "NumHAcceptors", "NumRotatableBonds",
    "HeavyAtomCount", "NumHeteroatoms", "NOCount", "NHOHCount",
    "RingCount", "NumAromaticRings", "NumAliphaticRings", "NumSaturatedRings",
    "FractionCSP3", "BertzCT", "LabuteASA", "BalabanJ",
    "FormalCharge", "NumValenceElectrons", "MaxPartialCharge", "MinPartialCharge",
]

def _artifact_get(model_artifact, key, default=None):
    return model_artifact.get(key, default) if isinstance(model_artifact, dict) else default

def _safe_descriptor_for_prediction(func, mol, default=np.nan):
    try:
        value = func(mol)
        if value is None or not np.isfinite(value):
            return default
        return value
    except Exception:
        return default

def _calculate_partial_charges_for_prediction(mol):
    mol_h = Chem.AddHs(mol)
    try:
        AllChem.ComputeGasteigerCharges(mol_h)
        charges = []
        for atom in mol_h.GetAtoms():
            charge = atom.GetProp("_GasteigerCharge")
            if charge not in ["nan", "-nan", "inf", "-inf"]:
                charges.append(float(charge))
        if not charges:
            return np.nan, np.nan
        return max(charges), min(charges)
    except Exception:
        return np.nan, np.nan

def _bitvect_to_array_for_prediction(bitvect, n_bits):
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(bitvect, arr)
    return arr

def _get_prediction_feature_metadata(model_artifact):
    continuous_cols = _artifact_get(
        model_artifact, "continuous_cols", DEFAULT_CONTINUOUS_COLS_FOR_INFERENCE
    )
    fingerprint_config = _artifact_get(
        model_artifact,
        "fingerprint_config",
        {"morgan_radius": 2, "morgan_bits": 2048},
    )
    fp_cols = _artifact_get(model_artifact, "fp_cols", None)
    if fp_cols is None:
        fp_cols = [
            f"MORGAN_{i}"
            for i in range(fingerprint_config.get("morgan_bits", 2048))
        ]
    return list(continuous_cols), list(fp_cols), dict(fingerprint_config)

def _calculate_features_for_prediction(smiles, fp_config):
    if pd.isna(smiles):
        return None
    mol = Chem.MolFromSmiles(str(smiles).strip())
    if mol is None:
        return None

    max_partial_charge, min_partial_charge = _calculate_partial_charges_for_prediction(mol)
    features = {
        "MolWt": Descriptors.MolWt(mol),
        "ExactMolWt": Descriptors.ExactMolWt(mol),
        "MolLogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "MolMR": Descriptors.MolMR(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        "HeavyAtomCount": Descriptors.HeavyAtomCount(mol),
        "NumHeteroatoms": Descriptors.NumHeteroatoms(mol),
        "NOCount": Descriptors.NOCount(mol),
        "NHOHCount": Descriptors.NHOHCount(mol),
        "RingCount": Descriptors.RingCount(mol),
        "NumAromaticRings": Descriptors.NumAromaticRings(mol),
        "NumAliphaticRings": Descriptors.NumAliphaticRings(mol),
        "NumSaturatedRings": Descriptors.NumSaturatedRings(mol),
        "FractionCSP3": Descriptors.FractionCSP3(mol),
        "BertzCT": Descriptors.BertzCT(mol),
        "LabuteASA": _safe_descriptor_for_prediction(Descriptors.LabuteASA, mol),
        "BalabanJ": _safe_descriptor_for_prediction(Descriptors.BalabanJ, mol),
        "FormalCharge": Chem.GetFormalCharge(mol),
        "NumValenceElectrons": Descriptors.NumValenceElectrons(mol),
        "MaxPartialCharge": max_partial_charge,
        "MinPartialCharge": min_partial_charge,
    }

    n_bits = fp_config.get("morgan_bits", 2048)
    radius = fp_config.get("morgan_radius", 2)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    arr = _bitvect_to_array_for_prediction(fp, n_bits)
    for i, value in enumerate(arr):
        features[f"MORGAN_{i}"] = int(value)
    return features

def _build_prediction_feature_table(smiles_df, model_artifact, smiles_col="smiles"):
    continuous_cols, fp_cols, fp_config = _get_prediction_feature_metadata(model_artifact)
    feature_cols = continuous_cols + fp_cols
    rows, valid_idx = [], []

    for idx, smiles in smiles_df[smiles_col].items():
        feats = _calculate_features_for_prediction(smiles, fp_config)
        if feats is not None:
            rows.append(feats)
            valid_idx.append(idx)

    if not rows:
        return pd.DataFrame(columns=feature_cols), smiles_df.iloc[[]].copy()

    X_new = pd.DataFrame(rows).reindex(columns=feature_cols, fill_value=0)
    valid_new = smiles_df.loc[valid_idx].copy()
    return X_new.reset_index(drop=True), valid_new.reset_index(drop=True)

def _applicability_domain_for_smiles(smiles_list, model_artifact):
    ad = _artifact_get(model_artifact, "applicability_domain", None)
    fp_config = _artifact_get(
        model_artifact,
        "fingerprint_config",
        {"morgan_radius": 2, "morgan_bits": 2048},
    )

    if not ad or not ad.get("reference_smiles"):
        return np.full(len(smiles_list), np.nan), np.full(len(smiles_list), False)

    reference_fps = []
    for s in ad["reference_smiles"]:
        mol = Chem.MolFromSmiles(str(s))
        if mol is not None:
            reference_fps.append(
                AllChem.GetMorganFingerprintAsBitVect(
                    mol,
                    int(fp_config["morgan_radius"]),
                    nBits=int(fp_config["morgan_bits"]),
                )
            )

    max_sims = []
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            max_sims.append(np.nan)
            continue

        fp = AllChem.GetMorganFingerprintAsBitVect(
            mol,
            int(fp_config["morgan_radius"]),
            nBits=int(fp_config["morgan_bits"]),
        )
        sims = DataStructs.BulkTanimotoSimilarity(fp, reference_fps)
        max_sims.append(float(max(sims)) if sims else np.nan)

    max_sims = np.asarray(max_sims, dtype=float)
    within_ad = max_sims >= float(ad["threshold"])
    return max_sims, within_ad

def predict_docking_score_from_smiles(smiles_df, model_artifact, smiles_col="smiles"):
    if smiles_col not in smiles_df.columns:
        raise ValueError(
            f"Column '{smiles_col}' not found. Available columns: {smiles_df.columns.tolist()}"
        )

    if isinstance(model_artifact, dict) and "model" in model_artifact:
        model = model_artifact["model"]
    elif hasattr(model_artifact, "predict"):
        model = model_artifact
    else:
        raise ValueError("Unsupported artifact format.")

    output = smiles_df.copy().reset_index(drop=True)
    output["input_order"] = np.arange(1, len(output) + 1)
    output["docking_score_predicted"] = np.nan
    output["max_tanimoto_to_ad_reference"] = np.nan
    output["within_applicability_domain"] = False
    output["target_name"] = _artifact_get(model_artifact, "target_metadata", {}).get("target_name")
    output["disease_name"] = _artifact_get(model_artifact, "target_metadata", {}).get("disease_name")
    output["docking_protocol_id"] = _artifact_get(model_artifact, "target_metadata", {}).get("docking_protocol_id")
    output["prediction_status"] = "invalid_smiles"

    X_new, valid_new = _build_prediction_feature_table(
        output, model_artifact, smiles_col
    )
    if len(valid_new) == 0:
        raise ValueError("No valid SMILES was found in the external CSV.")

    preds = model.predict(X_new)
    max_sims, within_ad = _applicability_domain_for_smiles(
        valid_new[smiles_col].tolist(), model_artifact
    )

    valid_positions = valid_new["input_order"].to_numpy() - 1
    output.loc[valid_positions, "docking_score_predicted"] = preds
    output.loc[valid_positions, "max_tanimoto_to_ad_reference"] = max_sims
    output.loc[valid_positions, "within_applicability_domain"] = within_ad
    output.loc[valid_positions, "prediction_status"] = "valid_smiles"

    return output.sort_values("input_order").reset_index(drop=True)

In [ ]:
print("=" * 60)
print("OPTIONAL EXTERNAL DOCKING-SCORE PREDICTION")
print("=" * 60)
print(f"Expected optional file: {SCREENING_CSV}")

if RUN_EXTERNAL_PREDICTION and not SCREENING_CSV.exists():
    raise FileNotFoundError(
        f"RUN_EXTERNAL_PREDICTION=True, but external screening file was not found: {SCREENING_CSV}"
    )

if RUN_EXTERNAL_PREDICTION:
    df_screening = pd.read_csv(SCREENING_CSV)
    df_screening.columns = df_screening.columns.str.strip()
    print("External file dimensions:", df_screening.shape)
    print("Detected columns:", df_screening.columns.tolist())
    display(df_screening.head())
else:
    df_screening = None
    print(
        "External screening skipped because the optional CSV was not found. "
        "Training, validation and model serialization are complete without it."
    )

In [ ]:
SMILES_COL_EXTERNAL = "smiles"

if RUN_EXTERNAL_PREDICTION:
    # joblib/pickle artifacts must only be loaded from trusted sources.
    artifact_for_inference = joblib.load(
        require_file(MODEL_PATH, "trusted trained model (.joblib)")
    )

    print("Loaded artifact:", MODEL_PATH)
    print("Artifact keys:", list(artifact_for_inference.keys()))

    predicoes = predict_docking_score_from_smiles(
        df_screening,
        artifact_for_inference,
        smiles_col=SMILES_COL_EXTERNAL,
    )
    display(predicoes.head(20))
else:
    artifact_for_inference = None
    predicoes = None

In [ ]:
if RUN_EXTERNAL_PREDICTION:
    print("=" * 60)
    print("SCREENING RESULT")
    print("=" * 60)
    print(f"Received molecules: {len(df_screening)}")
    print(
        "Molecules with valid SMILES and predictions:",
        (predicoes["prediction_status"] == "valid_smiles").sum(),
    )
    print(
        "Invalid SMILES:",
        (predicoes["prediction_status"] == "invalid_smiles").sum(),
    )
    display(predicoes.head(20))
    print("=" * 60)

In [ ]:
if RUN_EXTERNAL_PREDICTION:
    plt.figure(figsize=(8, 5))
    sns.histplot(
        predicoes["docking_score_predicted"].dropna(),
        kde=True,
        bins=30,
    )
    plt.xlabel("Predicted docking score")
    plt.ylabel("Frequency")
    plt.title("Distribution of predicted docking scores in external screening")
    plt.show()

In [ ]:
if RUN_EXTERNAL_PREDICTION:
    predicoes.to_csv(PREDICTIONS_CSV, index=False)
    announce_saved(PREDICTIONS_CSV, "screening result (CSV)")

If you want to start from an external prediction CSV already ranked by the model, use this alternative instead:

## Where are my results?

After a successful training run, the main files are inside `BASE_DIR`. The next cell lists the trained model, checksum, model card, target metadata, environment metadata, dependency lock and optional external predictions.

In [ ]:
output_catalog = [
    ("Trained model", MODEL_PATH, "model + preprocessing"),
    ("Model SHA-256", MODEL_SHA256_PATH, "artifact integrity"),
    ("Model card", MODEL_CARD_PATH, "scientific context and validation"),
    ("Target metadata", TARGET_METADATA_JSON, "disease/target/docking provenance"),
    ("Environment metadata", ENVIRONMENT_JSON, "versions, hashes and run config"),
    ("Dependency lock", REQUIREMENTS_LOCK, "exact Python package versions"),
]
if "PREDICTIONS_CSV" in globals(): output_catalog.append(("External predictions", PREDICTIONS_CSV, "predictions + AD"))
output_index = pd.DataFrame([{
    "result": name, "exists": Path(path).exists(), "path": str(Path(path).resolve()), "purpose": purpose
} for name, path, purpose in output_catalog])
print("RUN OUTPUT INDEX")
display(output_index)
print("Main output folder:", BASE_DIR.resolve())

## Public-release checklist

Before publishing a trained target model:

- complete the biological-target and disease metadata;
- report the exact protein structure and docking protocol;
- retain the dataset hash and software-environment record;
- publish held-out performance and applicability-domain coverage;
- publish the generated model card with the serialized model;
- state clearly that the endpoint is **predicted docking score**, not experimental affinity;
- document the provenance and license of any dataset distributed with the model.

The emphasis on understudied or neglected diseases is a project-level scientific priority; it does not change the statistical meaning of the docking-score model.

Publication-ready controls included:

- dummy-regression baseline;
- structural fold diagnostics;
- optimized-model out-of-fold metrics;
- group-aware bootstrap confidence intervals;
- Y-scrambling negative control;
- empirical error-versus-Tanimoto applicability-domain analysis;
- exact dependency lock;
- strict publication dependency checks.

Generative molecular design is intentionally outside the scope of this notebook.

For the public MolFlood repository, distribute this notebook together with `requirements.txt`, a tested tutorial dataset when available, a README with a minimal start-to-finish example, a chosen open-source license, and citation instructions (`CITATION.cff`).

Every released trained model should retain its MolFlood version, target, structure, docking protocol, validation results and applicability-domain metadata.

In [ ]:
# Run this before preparing a public release.
# PUBLICATION_MODE enforces complete essential campaign metadata.
public_release_metadata_issues = validate_target_metadata(
    TARGET_METADATA,
    strict=PUBLICATION_MODE,
)

if not public_release_metadata_issues:
    print("Campaign metadata is ready for public-release review.")